# Seam Residual Corrector v1 Colab Notebook

Готовый ноутбук для **training -> validation -> eval -> export -> verify_export**.

Локально нужно заранее собрать training bundle через `scripts/build_final_training_bundle.py`, загрузить `.tar.gz` на Google Drive и указать путь в параметрах ниже.

In [ ]:
# 0. PARAMS
from pathlib import Path
import os

DATASET_BUNDLE_DRIVE_PATH = '/content/drive/MyDrive/unet_seam/seam_residual_corrector_training_bundle.tar.gz'
DRIVE_RUNS_DIR = '/content/drive/MyDrive/unet_seam_runs'
RUN_NAME = 'seam_residual_corrector_v1_run001'
USE_RAMDISK = True
RAMDISK_SIZE_GB = 48
COPY_ARCHIVE_TO_RAM_FIRST = True
SYNC_INTERVAL_SEC = 180
TRAIN_BATCH_SIZE = 32
TRAIN_EPOCHS = 20
TRAIN_NUM_WORKERS = 16
VAL_BATCH_SIZE = 8
PRIMARY_CHECKPOINT = 'best_boundary_ciede2000.pt'
PROJECT_ROOT = Path('/content/seam_runtime')
LOCAL_OUTPUTS = PROJECT_ROOT / 'outputs'
LOCAL_CHECKPOINTS = LOCAL_OUTPUTS / 'checkpoints'
LOCAL_EVAL = LOCAL_OUTPUTS / 'eval_reports'
LOCAL_EXPORTS = LOCAL_OUTPUTS / 'exports'
LOCAL_LOGS = LOCAL_OUTPUTS / 'logs'
LOCAL_DATA_ROOT = Path('/content/dataset_bundle')
DRIVE_RUN_DIR = Path(DRIVE_RUNS_DIR) / RUN_NAME
DRIVE_CKPT_DIR = DRIVE_RUN_DIR / 'checkpoints'
DRIVE_EVAL_DIR = DRIVE_RUN_DIR / 'eval_reports'
DRIVE_EXPORT_DIR = DRIVE_RUN_DIR / 'exports'
DRIVE_LOG_DIR = DRIVE_RUN_DIR / 'logs'


In [ ]:
# 1. MOUNT DRIVE
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted')


In [ ]:
# 2. INSTALL / RUNTIME CHECKS
import os, sys, subprocess, importlib.util, platform, json
pkgs = ['pyyaml','scipy','scikit-image','safetensors','tqdm','lpips','psutil']
subprocess.run(['apt-get', 'update', '-qq'], check=False)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'pigz'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs, check=True)
import torch
device = 'cuda' if torch.cuda.is_available() else 'mps' if getattr(torch.backends, 'mps', None) and torch.backends.mps.is_available() else 'cpu'
print(json.dumps({'python': sys.executable, 'platform': platform.platform(), 'torch': torch.__version__, 'device': device}, ensure_ascii=False))


In [ ]:
# 3. OPTIONAL RAMDISK
import os, psutil, subprocess, shutil
RAM_ROOT = Path('/content/ramdisk')
if USE_RAMDISK:
    mem = psutil.virtual_memory()
    eff = min(RAMDISK_SIZE_GB, max(8, int((mem.available / (1024**3)) * 0.5)))
    RAM_ROOT.mkdir(parents=True, exist_ok=True)
    mounted = subprocess.run(['mountpoint', str(RAM_ROOT)], capture_output=True).returncode == 0
    if not mounted:
        subprocess.run(['mount', '-t', 'tmpfs', '-o', f'size={eff}G,mode=777', 'tmpfs', str(RAM_ROOT)], check=True)
    DATA_ROOT = RAM_ROOT / 'dataset_bundle'
else:
    DATA_ROOT = LOCAL_DATA_ROOT
DATA_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_OUTPUTS.mkdir(parents=True, exist_ok=True)
for p in [LOCAL_CHECKPOINTS, LOCAL_EVAL, LOCAL_EXPORTS, LOCAL_LOGS]:
    p.mkdir(parents=True, exist_ok=True)
print('DATA_ROOT =', DATA_ROOT)


In [ ]:
# 4. EXTRACT DATASET BUNDLE FROM DRIVE
import shutil, tarfile, time
from tqdm.auto import tqdm

src = Path(DATASET_BUNDLE_DRIVE_PATH)
if not src.exists():
    raise FileNotFoundError(src)
archive_local = src
if COPY_ARCHIVE_TO_RAM_FIRST and USE_RAMDISK:
    archive_local = RAM_ROOT / src.name
    with src.open('rb') as fin, archive_local.open('wb') as fout:
        total = src.stat().st_size
        with tqdm(total=total, unit='B', unit_scale=True, desc='copy_archive') as pbar:
            while True:
                chunk = fin.read(8 * 1024 * 1024)
                if not chunk:
                    break
                fout.write(chunk)
                pbar.update(len(chunk))
if DATA_ROOT.exists():
    shutil.rmtree(DATA_ROOT)
DATA_ROOT.mkdir(parents=True, exist_ok=True)
with tarfile.open(archive_local, 'r:*') as tf:
    members = tf.getmembers()
    for m in tqdm(members, desc='extract_bundle', dynamic_ncols=True):
        tf.extract(m, DATA_ROOT)
print('bundle extracted to', DATA_ROOT)


In [ ]:
# 5. WRITE PROJECT FILES INTO RUNTIME
import json, os
PROJECT_FILES = {
  "configs/model_resunet_v1.yaml": "model:\n  name: seam_residual_corrector_v1\n  in_channels: 5\n  out_channels: 3\n  base_channels: 32\n  depth: 4\n  groups: 8\n  residual_cap: 0.3\n  residual_mode: full\n  low_freq_sigma: 6.0\n  gap_film: true\n  lf_branch: true\nstrip:\n  height: 1024\n  outer_width: 128\n  inner_width_default: 128\n  supported_inner_widths: [96, 128, 160, 192]\n  seam_jitter_train_px: 16\nloss:\n  boundary_band_px: 24\n  lowfreq_sigmas: [2.0, 4.0, 8.0, 16.0, 32.0]\n",
  "configs/train_synth_v1.yaml": "seed: 42\ndataset:\n  source_manifest: manifests/input_raw_manifest.jsonl\n  source_root: data/source_images\n  cache_root: outputs/strip_cache\n  train_cache_manifest: manifests/strip_train_cache.jsonl\n  val_cache_manifest: manifests/strip_val_cache.jsonl\n  strips_per_image: 25\n  strip_height: 1024\n  outer_width: 128\n  supported_inner_widths: [96, 128, 160, 192]\n  seam_jitter_px: 16\n  boundary_band_px: 24\n  weighted_scene_tags: [sky, skin, gradient, night, wall]\nmodel:\n  residual_mode: full\n  low_freq_sigma: 6.0\ntrain:\n  seed: 42\n  optimizer: AdamW\n  lr: 2.0e-4\n  weight_decay: 1.0e-4\n  betas: [0.9, 0.999]\n  grad_clip: 1.0\n  precision: bf16\n  batch_size: 16\n  num_epochs: 20\n  num_workers: 0\nscheduler:\n  type: cosine_with_warmup\n  warmup_steps: 1000\n  min_lr: 1.0e-6\nema:\n  enabled: true\n  decay: 0.999\nplateau:\n  patience_epochs: 3\n  reduce_factor: 0.5\n  min_lr_stop: 1.0e-6\nearly_stop:\n  patience_epochs: 5\n  metric: val_boundary_ciede2000\nlogging:\n  tensorboard: true\n  log_dir: outputs/logs/tensorboard\n  log_interval: 20\n  # In Colab / subprocess, tqdm is hidden: print a JSON line every N train steps (set 0 to disable).\n  console_log_interval: 25\nviewer_cache:\n  root: outputs/strip_cache/val\n",
  "configs/eval_v1.yaml": "checkpoint: outputs/checkpoints/best_boundary_ciede2000.pt\nreport_root: outputs/eval_reports\nbatch_size: 8\nbootstrap_samples: 200\nprimary_metric: boundary_ciede2000\ncache_root: outputs/strip_cache\nval_cache_manifest: manifests/strip_val_cache.jsonl\n",
  "configs/export_v1.yaml": "checkpoint: outputs/checkpoints/best_boundary_ciede2000.pt\nexport_root: outputs/exports\nmodel_name: seam_residual_corrector_v1\nstrength_default: 1.0\n",
  "src/__init__.py": "\"\"\"Seam residual corrector package.\"\"\"\n",
  "src/data/__init__.py": "\"\"\"Data pipeline modules.\"\"\"\n",
  "src/data/manifest.py": "from __future__ import annotations\n\nimport json\nfrom pathlib import Path\nfrom typing import Iterable\n\n\ndef _to_path(path: str | Path) -> Path:\n    return path if isinstance(path, Path) else Path(path)\n\n\ndef append_jsonl(path: Path, rows: Iterable[dict]) -> None:\n    path = _to_path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with path.open(\"a\", encoding=\"utf-8\") as handle:\n        for row in rows:\n            handle.write(json.dumps(row, ensure_ascii=False) + \"\\n\")\n\n\ndef write_jsonl(path: Path, rows: Iterable[dict]) -> None:\n    path = _to_path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with path.open(\"w\", encoding=\"utf-8\") as handle:\n        for row in rows:\n            handle.write(json.dumps(row, ensure_ascii=False) + \"\\n\")\n\n\ndef read_jsonl(path: str | Path) -> list[dict]:\n    path = _to_path(path)\n    if not path.exists():\n        return []\n    rows: list[dict] = []\n    with path.open(\"r\", encoding=\"utf-8\") as handle:\n        for line in handle:\n            line = line.strip()\n            if line:\n                rows.append(json.loads(line))\n    return rows\n",
  "src/data/preprocess.py": "from __future__ import annotations\n\nimport re\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom typing import Optional\n\nfrom src.utils.image_io import center_crop_square, open_rgb_image, resize_square, save_png, sha256_file\nfrom src.utils.phash import compute_phash64\n\n\nCRITICAL_SCENE_TAGS = {\n    \"sky\",\n    \"skin\",\n    \"gradient\",\n    \"night\",\n    \"wall\",\n    \"water\",\n    \"glass\",\n    \"architecture\",\n    \"leaves\",\n}\n\n\n@dataclass\nclass PreparedSource:\n    row: dict\n    excluded_reason: Optional[dict] = None\n\n\ndef extract_scene_tags(caption: str) -> list[str]:\n    tokens = {token.lower() for token in re.findall(r\"[a-zA-Z_]+\", caption)}\n    return sorted(token for token in CRITICAL_SCENE_TAGS if token in tokens)\n\n\ndef prepare_single_source(\n    path: Path,\n    output_dir: Path,\n    sample_id: str,\n) -> PreparedSource:\n    if path.suffix.lower() not in {\".jpg\", \".jpeg\", \".png\"}:\n        return PreparedSource({}, {\"path\": str(path), \"reason\": \"unsupported_extension\"})\n    try:\n        image = open_rgb_image(path)\n    except Exception as exc:\n        return PreparedSource({}, {\"path\": str(path), \"reason\": str(exc)})\n    if min(image.size) < 512:\n        return PreparedSource({}, {\"path\": str(path), \"reason\": \"too_small\"})\n    image = resize_square(center_crop_square(image), 1024)\n    out_path = output_dir / f\"{sample_id}.png\"\n    save_png(image, out_path)\n    caption_path = path.with_suffix(\".txt\")\n    caption = caption_path.read_text(encoding=\"utf-8\").strip() if caption_path.exists() else \"\"\n    row = {\n        \"id\": sample_id,\n        \"source_path\": str(out_path).replace(\"\\\\\", \"/\"),\n        \"original_path\": str(path).replace(\"\\\\\", \"/\"),\n        \"caption_path\": str(caption_path).replace(\"\\\\\", \"/\") if caption_path.exists() else None,\n        \"caption\": caption,\n        \"scene_tags\": extract_scene_tags(caption),\n        \"phash64\": compute_phash64(out_path),\n        \"cluster_id\": None,\n        \"split\": None,\n        \"width\": 1024,\n        \"height\": 1024,\n        \"has_icc\": bool(image.info.get(\"icc_profile\")),\n        \"sha256\": sha256_file(out_path),\n        \"source_domain\": \"photo\",\n    }\n    return PreparedSource(row=row)\n",
  "src/data/strip_geometry.py": "from __future__ import annotations\n\nimport math\nfrom dataclasses import dataclass\nfrom typing import Literal\n\nimport torch\nimport torch.nn.functional as F\n\n\nSide = Literal[\"left\", \"right\", \"top\", \"bottom\"]\n\n\n@dataclass(frozen=True)\nclass StripSpec:\n    strip_height: int = 1024\n    outer_width: int = 128\n    inner_width: int = 128\n    seam_jitter_px: int = 16\n\n    @property\n    def width(self) -> int:\n        return self.outer_width + self.inner_width\n\n\ndef canonicalize_strip(strip: torch.Tensor, side: Side) -> torch.Tensor:\n    if side == \"left\":\n        return strip\n    if side == \"right\":\n        return torch.flip(strip, dims=(-1,))\n    if side == \"top\":\n        return torch.rot90(strip, k=1, dims=(-2, -1))\n    if side == \"bottom\":\n        return torch.rot90(strip, k=3, dims=(-2, -1))\n    raise ValueError(f\"unsupported side: {side}\")\n\n\ndef decanonicalize_strip(strip: torch.Tensor, side: Side) -> torch.Tensor:\n    if side == \"left\":\n        return strip\n    if side == \"right\":\n        return torch.flip(strip, dims=(-1,))\n    if side == \"top\":\n        return torch.rot90(strip, k=3, dims=(-2, -1))\n    if side == \"bottom\":\n        return torch.rot90(strip, k=1, dims=(-2, -1))\n    raise ValueError(f\"unsupported side: {side}\")\n\n\ndef make_inner_mask(height: int, width: int, seam_x: int) -> torch.Tensor:\n    xs = torch.arange(width, dtype=torch.float32).view(1, 1, 1, width)\n    return (xs >= seam_x).to(torch.float32).expand(1, 1, height, width)\n\n\ndef make_distance_to_seam(height: int, width: int, seam_x: int) -> torch.Tensor:\n    xs = torch.arange(width, dtype=torch.float32).view(1, 1, 1, width)\n    max_distance = max(seam_x, width - seam_x)\n    return (xs.sub(float(seam_x)).abs() / float(max_distance)).expand(1, 1, height, width)\n\n\ndef make_boundary_band_mask(height: int, width: int, seam_x: int, band_px: int = 24) -> torch.Tensor:\n    xs = torch.arange(width, dtype=torch.float32).view(1, 1, 1, width)\n    return (xs.sub(float(seam_x)).abs() <= band_px).to(torch.float32).expand(1, 1, height, width)\n\n\ndef build_decay_mask(height: int, width: int, seam_x: int, inner_width: int) -> torch.Tensor:\n    xs = torch.arange(width, dtype=torch.float32).view(1, 1, 1, width)\n    t = ((xs - seam_x) / max(inner_width, 1)).clamp(0.0, 1.0)\n    decay = 0.5 * (1.0 + torch.cos(math.pi * t))\n    decay = torch.where(xs < seam_x, torch.zeros_like(decay), decay)\n    return decay.expand(1, 1, height, width)\n\n\ndef _replicate_pad_strip(strip: torch.Tensor, target_width: int) -> tuple[torch.Tensor, int]:\n    edge_padded = max(target_width - strip.shape[-1], 0)\n    if edge_padded == 0:\n        return strip, 0\n    padded = F.pad(strip, (0, edge_padded, 0, 0), mode=\"replicate\")\n    return padded, edge_padded\n\n\ndef extract_side_strip(\n    image: torch.Tensor,\n    bbox: tuple[int, int, int, int],\n    side: Side,\n    spec: StripSpec,\n) -> tuple[torch.Tensor, dict]:\n    if image.ndim != 3:\n        raise ValueError(\"expected CHW image tensor\")\n    _, h, w = image.shape\n    x0, y0, x1, y1 = bbox\n    if side == \"left\":\n        y_center = (y0 + y1) // 2\n        y_start = max(0, min(h - spec.strip_height, y_center - spec.strip_height // 2))\n        outer = image[:, y_start : y_start + spec.strip_height, max(0, x0 - spec.outer_width) : x0]\n        inner = image[:, y_start : y_start + spec.strip_height, x0 : min(w, x0 + spec.inner_width)]\n        strip = torch.cat([outer, inner], dim=-1)\n        strip, edge_padded = _replicate_pad_strip(strip, spec.width)\n    elif side == \"right\":\n        y_center = (y0 + y1) // 2\n        y_start = max(0, min(h - spec.strip_height, y_center - spec.strip_height // 2))\n        inner = image[:, y_start : y_start + spec.strip_height, max(0, x1 - spec.inner_width) : x1]\n        outer = image[:, y_start : y_start + spec.strip_height, x1 : min(w, x1 + spec.outer_width)]\n        strip = torch.cat([inner, outer], dim=-1)\n        strip, edge_padded = _replicate_pad_strip(strip, spec.width)\n        strip = canonicalize_strip(strip, \"right\")\n    elif side == \"top\":\n        x_center = (x0 + x1) // 2\n        x_start = max(0, min(w - spec.strip_height, x_center - spec.strip_height // 2))\n        outer = image[:, max(0, y0 - spec.outer_width) : y0, x_start : x_start + spec.strip_height]\n        inner = image[:, y0 : min(h, y0 + spec.inner_width), x_start : x_start + spec.strip_height]\n        strip = torch.cat([outer, inner], dim=-2)\n        strip = F.pad(strip, (0, 0, 0, max(spec.width - strip.shape[-2], 0)), mode=\"replicate\")\n        edge_padded = max(spec.width - strip.shape[-2], 0)\n        strip = canonicalize_strip(strip, \"top\")\n    elif side == \"bottom\":\n        x_center = (x0 + x1) // 2\n        x_start = max(0, min(w - spec.strip_height, x_center - spec.strip_height // 2))\n        inner = image[:, max(0, y1 - spec.inner_width) : y1, x_start : x_start + spec.strip_height]\n        outer = image[:, y1 : min(h, y1 + spec.outer_width), x_start : x_start + spec.strip_height]\n        strip = torch.cat([inner, outer], dim=-2)\n        strip = F.pad(strip, (0, 0, 0, max(spec.width - strip.shape[-2], 0)), mode=\"replicate\")\n        edge_padded = max(spec.width - strip.shape[-2], 0)\n        strip = canonicalize_strip(strip, \"bottom\")\n    else:\n        raise ValueError(f\"unsupported side: {side}\")\n    return strip, {\"edge_padded_pixels\": int(edge_padded)}\n\n\ndef validate_roundtrip(strip: torch.Tensor, side: Side) -> bool:\n    return torch.equal(decanonicalize_strip(canonicalize_strip(strip, side), side), strip)\n",
  "src/data/corruptions.py": "from __future__ import annotations\n\nimport math\nfrom dataclasses import dataclass\n\nimport torch\nimport torch.nn.functional as F\n\n\nGROUPS = {\n    \"A\": [\"brightness\", \"exposure\", \"gain\"],\n    \"B\": [\"contrast\", \"gamma\"],\n    \"C\": [\"temperature\", \"tint\", \"channel_gains\", \"saturation\"],\n    \"D\": [\"shadow\", \"highlight\", \"midtone\"],\n    \"E\": [\"brightness_gradient\", \"color_gradient\", \"illumination_poly\", \"vignette\"],\n    \"F\": [\"blur\", \"jpeg_like\", \"noise\", \"microcontrast\"],\n}\n\n\n@dataclass\nclass CorruptionResult:\n    image: torch.Tensor\n    ops: list[str]\n\n\ndef _rgb_to_luma(x: torch.Tensor) -> torch.Tensor:\n    weights = torch.tensor([0.2126, 0.7152, 0.0722], device=x.device, dtype=x.dtype).view(1, 3, 1, 1)\n    return (x * weights).sum(dim=1, keepdim=True)\n\n\ndef _apply_gamma(x: torch.Tensor, gamma: float) -> torch.Tensor:\n    return x.clamp(1e-6, 1.0).pow(gamma)\n\n\ndef _gaussian_kernel1d(sigma: float, device: torch.device, dtype: torch.dtype) -> torch.Tensor:\n    radius = max(1, int(round(sigma * 3)))\n    xs = torch.arange(-radius, radius + 1, device=device, dtype=dtype)\n    kernel = torch.exp(-(xs**2) / (2 * sigma * sigma))\n    kernel /= kernel.sum()\n    return kernel\n\n\ndef _gaussian_blur(x: torch.Tensor, sigma: float) -> torch.Tensor:\n    if sigma <= 0:\n        return x\n    kernel = _gaussian_kernel1d(sigma, x.device, x.dtype)\n    pad = min(kernel.numel() // 2, max(1, x.shape[-1] // 2 - 1), max(1, x.shape[-2] // 2 - 1))\n    if pad * 2 + 1 != kernel.numel():\n        kernel = kernel[kernel.numel() // 2 - pad : kernel.numel() // 2 + pad + 1]\n        kernel = kernel / kernel.sum()\n    kernel_x = kernel.view(1, 1, 1, -1).repeat(x.shape[1], 1, 1, 1)\n    kernel_y = kernel.view(1, 1, -1, 1).repeat(x.shape[1], 1, 1, 1)\n    x = F.conv2d(F.pad(x, (pad, pad, 0, 0), mode=\"reflect\"), kernel_x, groups=x.shape[1])\n    x = F.conv2d(F.pad(x, (0, 0, pad, pad), mode=\"reflect\"), kernel_y, groups=x.shape[1])\n    return x\n\n\ndef _field(shape: torch.Size, magnitude: float, generator: torch.Generator) -> torch.Tensor:\n    _, _, h, w = shape\n    yy = torch.linspace(-1.0, 1.0, h).view(1, 1, h, 1)\n    xx = torch.linspace(-1.0, 1.0, w).view(1, 1, 1, w)\n    ax = torch.rand(1, generator=generator).item() * magnitude\n    ay = torch.rand(1, generator=generator).item() * magnitude\n    return ax * xx + ay * yy\n\n\ndef apply_random_corruptions(inner: torch.Tensor, generator: torch.Generator) -> CorruptionResult:\n    x = inner.clone()\n    ops: list[str] = []\n    candidates = [\n        \"brightness\",\n        \"exposure\",\n        \"gain\",\n        \"contrast\",\n        \"gamma\",\n        \"temperature\",\n        \"tint\",\n        \"channel_gains\",\n        \"saturation\",\n        \"shadow\",\n        \"highlight\",\n        \"midtone\",\n        \"brightness_gradient\",\n        \"color_gradient\",\n        \"illumination_poly\",\n        \"vignette\",\n        \"blur\",\n        \"noise\",\n        \"microcontrast\",\n    ]\n    n_ops = int(torch.multinomial(torch.tensor([0.2, 0.4, 0.3, 0.1]), 1, generator=generator).item()) + 2\n    chosen: list[str] = []\n    while len(chosen) < n_ops:\n        idx = int(torch.randint(0, len(candidates), (1,), generator=generator).item())\n        name = candidates[idx]\n        if name not in chosen:\n            chosen.append(name)\n    if not any(name in GROUPS[\"A\"] + GROUPS[\"B\"] + GROUPS[\"C\"] for name in chosen):\n        chosen[0] = \"exposure\"\n    if not any(name in GROUPS[\"D\"] for name in chosen):\n        chosen[min(1, len(chosen) - 1)] = \"midtone\"\n    for name in chosen:\n        if name == \"brightness\":\n            delta = torch.empty(1).uniform_(-0.08, 0.08, generator=generator).item()\n            x = x + delta\n        elif name == \"exposure\":\n            ev = torch.empty(1).uniform_(-0.3, 0.5, generator=generator).item()\n            x = x * (2.0**ev)\n        elif name == \"gain\":\n            gain = torch.empty(1).uniform_(0.85, 1.2, generator=generator).item()\n            x = x * gain\n        elif name == \"contrast\":\n            contrast = torch.empty(1).uniform_(0.8, 1.25, generator=generator).item()\n            mean = x.mean(dim=(-2, -1), keepdim=True)\n            x = (x - mean) * contrast + mean\n        elif name == \"gamma\":\n            gamma = torch.empty(1).uniform_(0.85, 1.2, generator=generator).item()\n            x = _apply_gamma(x, gamma)\n        elif name == \"temperature\":\n            t = torch.empty(1).uniform_(-0.06, 0.06, generator=generator).item()\n            x[:, 0:1] += t\n            x[:, 2:3] -= t\n        elif name == \"tint\":\n            t = torch.empty(1).uniform_(-0.05, 0.05, generator=generator).item()\n            x[:, 1:2] += t\n        elif name == \"channel_gains\":\n            gains = torch.empty((1, 3, 1, 1)).uniform_(0.9, 1.1, generator=generator)\n            x = x * gains\n        elif name == \"saturation\":\n            sat = torch.empty(1).uniform_(0.75, 1.35, generator=generator).item()\n            luma = _rgb_to_luma(x)\n            x = luma + (x - luma) * sat\n        elif name == \"shadow\":\n            amount = torch.empty(1).uniform_(0.0, 0.12, generator=generator).item()\n            x = x + (1.0 - x) * amount * (1.0 - x).pow(2)\n        elif name == \"highlight\":\n            amount = torch.empty(1).uniform_(0.0, 0.12, generator=generator).item()\n            x = x - amount * x.pow(2)\n        elif name == \"midtone\":\n            amount = torch.empty(1).uniform_(-0.08, 0.08, generator=generator).item()\n            x = x + amount * torch.sin(x * math.pi)\n        elif name == \"brightness_gradient\":\n            x = x + _field(x.shape, 0.12, generator)\n        elif name == \"color_gradient\":\n            field = _field(x.shape, 0.12, generator)\n            x[:, 0:1] += field\n            x[:, 2:3] -= field * 0.8\n        elif name == \"illumination_poly\":\n            field = _field(x.shape, 0.10, generator)\n            x = x * (1.0 + field)\n        elif name == \"vignette\":\n            _, _, h, w = x.shape\n            yy = torch.linspace(-1.0, 1.0, h).view(1, 1, h, 1)\n            xx = torch.linspace(-1.0, 1.0, w).view(1, 1, 1, w)\n            rr = (xx * xx + yy * yy).sqrt()\n            scale = torch.empty(1).uniform_(0.0, 0.15, generator=generator).item()\n            x = x * (1.0 - rr * scale)\n        elif name == \"blur\":\n            sigma = torch.empty(1).uniform_(0.0, 1.5, generator=generator).item()\n            x = _gaussian_blur(x, sigma)\n        elif name == \"noise\":\n            sigma = torch.empty(1).uniform_(0.0, 0.01, generator=generator).item()\n            x = x + torch.randn_like(x, generator=generator) * sigma\n        elif name == \"microcontrast\":\n            amount = torch.empty(1).uniform_(0.0, 0.1, generator=generator).item()\n            blur = _gaussian_blur(x, 1.0)\n            x = x + (x - blur) * amount\n        ops.append(name)\n    return CorruptionResult(x.clamp(0.0, 1.0), ops)\n",
  "src/data/synthetic_strip_dataset.py": "from __future__ import annotations\n\nimport random\nfrom dataclasses import dataclass\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nimport torch.nn.functional as F\nfrom PIL import Image\nfrom torch.utils.data import Dataset\n\nfrom src.data.corruptions import apply_random_corruptions\nfrom src.data.manifest import read_jsonl\nfrom src.data.strip_geometry import (\n    StripSpec,\n    build_decay_mask,\n    canonicalize_strip,\n    make_boundary_band_mask,\n    make_distance_to_seam,\n    make_inner_mask,\n)\n\n\n@dataclass(frozen=True)\nclass SampleConfig:\n    axis: str\n    side: str\n    seam_x_frac: float\n    flip_h: bool\n    rotation_k: int\n    seam_jitter_px: int\n    inner_width: int\n\n\nclass SyntheticStripDataset(Dataset):\n    def __init__(\n        self,\n        manifest_path: Path,\n        strips_per_image: int = 25,\n        split: str | None = None,\n        seed: int = 42,\n        spec: StripSpec | None = None,\n        boundary_band_px: int = 24,\n    ) -> None:\n        self.rows = [row for row in read_jsonl(manifest_path) if not split or row.get(\"split\") == split]\n        self.strips_per_image = strips_per_image\n        self.seed = seed\n        self.spec = spec or StripSpec()\n        self.boundary_band_px = boundary_band_px\n        self.inner_widths = [96, 128, 160, 192]\n        self.base_variants = self._build_base_variants()\n\n    def _build_base_variants(self) -> list[SampleConfig]:\n        variants: list[SampleConfig] = []\n        seam_positions = np.linspace(0.25, 0.75, 8)\n        for axis in (\"vertical\", \"horizontal\"):\n            sides = (\"left\", \"right\") if axis == \"vertical\" else (\"top\", \"bottom\")\n            for frac in seam_positions:\n                for flip_h in (False, True):\n                    for rotation_k in range(4):\n                        side = sides[(rotation_k + int(flip_h)) % len(sides)]\n                        variants.append(\n                            SampleConfig(\n                                axis=axis,\n                                side=side,\n                                seam_x_frac=float(frac),\n                                flip_h=flip_h,\n                                rotation_k=rotation_k,\n                                seam_jitter_px=0,\n                                inner_width=128,\n                            )\n                        )\n        return variants\n\n    def __len__(self) -> int:\n        return len(self.rows) * self.strips_per_image\n\n    def _config_for_index(self, idx: int) -> SampleConfig:\n        rng = random.Random(self.seed + idx)\n        image_slot = idx // max(self.strips_per_image, 1)\n        epoch_variants = self.base_variants.copy()\n        rng.shuffle(epoch_variants)\n        base = epoch_variants[idx % min(len(epoch_variants), self.strips_per_image) if self.strips_per_image <= len(epoch_variants) else idx % len(epoch_variants)]\n        del image_slot\n        return SampleConfig(\n            axis=base.axis,\n            side=base.side,\n            seam_x_frac=base.seam_x_frac,\n            flip_h=base.flip_h,\n            rotation_k=base.rotation_k,\n            seam_jitter_px=rng.randint(-self.spec.seam_jitter_px, self.spec.seam_jitter_px),\n            inner_width=rng.choice(self.inner_widths),\n        )\n\n    def _load_image(self, row: dict) -> torch.Tensor:\n        arr = np.asarray(Image.open(row[\"source_path\"]).convert(\"RGB\"), dtype=np.float32) / 255.0\n        return torch.from_numpy(arr).permute(2, 0, 1)\n\n    def _augment_source_image(self, image: torch.Tensor, cfg: SampleConfig) -> torch.Tensor:\n        if cfg.flip_h:\n            image = torch.flip(image, dims=(-1,))\n        if cfg.rotation_k:\n            image = torch.rot90(image, k=cfg.rotation_k, dims=(-2, -1))\n        return image\n\n    def _extract_clean_strip(self, image: torch.Tensor, cfg: SampleConfig) -> torch.Tensor:\n        h, w = image.shape[-2:]\n        spec = StripSpec(strip_height=self.spec.strip_height, outer_width=self.spec.outer_width, inner_width=cfg.inner_width, seam_jitter_px=self.spec.seam_jitter_px)\n        if cfg.axis == \"vertical\":\n            seam_x = int(round(cfg.seam_x_frac * w))\n            seam_x = min(max(spec.outer_width, seam_x), w - spec.inner_width)\n            y_start = max(0, min(h - spec.strip_height, h // 2 - spec.strip_height // 2))\n            outer = image[:, y_start : y_start + spec.strip_height, seam_x - spec.outer_width : seam_x]\n            inner = image[:, y_start : y_start + spec.strip_height, seam_x : seam_x + cfg.inner_width]\n            strip = torch.cat([outer, inner], dim=-1)\n            if cfg.side == \"right\":\n                strip = canonicalize_strip(torch.flip(strip, dims=(-1,)), \"right\")\n        else:\n            seam_y = int(round(cfg.seam_x_frac * h))\n            seam_y = min(max(spec.outer_width, seam_y), h - cfg.inner_width)\n            x_start = max(0, min(w - spec.strip_height, w // 2 - spec.strip_height // 2))\n            outer = image[:, seam_y - spec.outer_width : seam_y, x_start : x_start + spec.strip_height]\n            inner = image[:, seam_y : seam_y + cfg.inner_width, x_start : x_start + spec.strip_height]\n            strip = torch.cat([outer, inner], dim=-2)\n            strip = canonicalize_strip(strip, cfg.side)\n        return strip\n\n    def __getitem__(self, idx: int) -> dict:\n        row = self.rows[idx // self.strips_per_image]\n        cfg = self._config_for_index(idx)\n        image = self._augment_source_image(self._load_image(row), cfg)\n        clean_strip = self._extract_clean_strip(image, cfg)\n        if clean_strip.shape[-2:] != (self.spec.strip_height, self.spec.outer_width + cfg.inner_width):\n            raise RuntimeError(\"unexpected strip shape\")\n        seam_x = self.spec.outer_width + cfg.seam_jitter_px\n        pad_left = max(0, -cfg.seam_jitter_px)\n        pad_right = max(0, cfg.seam_jitter_px)\n        clean_strip = F.pad(clean_strip, (pad_left, pad_right, 0, 0), mode=\"replicate\")\n        clean_strip = clean_strip[..., : self.spec.strip_height, : self.spec.outer_width + cfg.inner_width]\n        target = clean_strip.clone()\n        input_rgb = clean_strip.unsqueeze(0)\n        inner = input_rgb[..., self.spec.outer_width :]\n        corrupted = apply_random_corruptions(inner, torch.Generator().manual_seed(self.seed + idx))\n        input_rgb[..., self.spec.outer_width :] = corrupted.image\n        mask = make_inner_mask(self.spec.strip_height, self.spec.outer_width + cfg.inner_width, seam_x)\n        distance = make_distance_to_seam(self.spec.strip_height, self.spec.outer_width + cfg.inner_width, seam_x)\n        boundary = make_boundary_band_mask(self.spec.strip_height, self.spec.outer_width + cfg.inner_width, seam_x, self.boundary_band_px)\n        decay = build_decay_mask(self.spec.strip_height, self.spec.outer_width + cfg.inner_width, seam_x, cfg.inner_width)\n        sample = {\n            \"input\": torch.cat([input_rgb.squeeze(0), mask.squeeze(0), distance.squeeze(0)], dim=0),\n            \"target\": target,\n            \"input_rgb\": input_rgb.squeeze(0),\n            \"mask\": mask.squeeze(0),\n            \"distance\": distance.squeeze(0),\n            \"inner_region_mask\": mask.squeeze(0),\n            \"boundary_band_mask\": boundary.squeeze(0),\n            \"decay_mask\": decay.squeeze(0),\n            \"meta\": {\n                \"image_id\": row[\"id\"],\n                \"axis\": cfg.axis,\n                \"side\": cfg.side,\n                \"rotation_k\": cfg.rotation_k,\n                \"flip_h\": cfg.flip_h,\n                \"seam_jitter_px\": cfg.seam_jitter_px,\n                \"inner_width\": cfg.inner_width,\n                \"edge_padded_pixels\": 0,\n                \"ops\": corrupted.ops,\n                \"scene_tags\": row.get(\"scene_tags\", []),\n                \"split\": row.get(\"split\"),\n                \"cluster_id\": row.get(\"cluster_id\"),\n                \"seam_x_frac_in_source\": cfg.seam_x_frac,\n            },\n        }\n        return sample\n\n\ndef collate_strip_batch(samples: list[dict]) -> dict:\n    if not samples:\n        raise ValueError(\"empty batch\")\n    tensor_keys = [key for key, value in samples[0].items() if isinstance(value, torch.Tensor)]\n    batch: dict = {}\n    max_w = max(sample[\"input\"].shape[-1] for sample in samples)\n    for key in tensor_keys:\n        items = []\n        for sample in samples:\n            tensor = sample[key]\n            pad_w = max_w - tensor.shape[-1]\n            if pad_w > 0:\n                mode = \"replicate\" if tensor.shape[0] in {3, 5} else \"constant\"\n                if tensor.ndim == 3:\n                    if mode == \"constant\":\n                        tensor = F.pad(tensor, (0, pad_w, 0, 0), mode=mode, value=0.0)\n                    else:\n                        tensor = F.pad(tensor, (0, pad_w, 0, 0), mode=mode)\n                else:\n                    raise RuntimeError(f\"unexpected tensor rank for key={key}\")\n            items.append(tensor)\n        batch[key] = torch.stack(items, dim=0)\n    batch[\"meta\"] = [sample[\"meta\"] for sample in samples]\n    return batch\n",
  "src/data/cached_strip_dataset.py": "from __future__ import annotations\n\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom PIL import Image\nfrom torch.utils.data import Dataset\n\nfrom src.data.manifest import read_jsonl\nfrom src.data.strip_geometry import build_decay_mask, make_boundary_band_mask\n\n\ndef _load_rgb(path: Path) -> torch.Tensor:\n    arr = np.asarray(Image.open(path).convert(\"RGB\"), dtype=np.float32) / 255.0\n    return torch.from_numpy(arr).permute(2, 0, 1)\n\n\ndef _load_gray(path: Path) -> torch.Tensor:\n    arr = np.asarray(Image.open(path).convert(\"L\"), dtype=np.float32) / 255.0\n    return torch.from_numpy(arr).unsqueeze(0)\n\n\nclass CachedStripDataset(Dataset):\n    def __init__(self, manifest_path: str | Path, cache_root: str | Path) -> None:\n        self.rows = read_jsonl(manifest_path)\n        self.cache_root = Path(cache_root)\n\n    def __len__(self) -> int:\n        return len(self.rows)\n\n    def __getitem__(self, idx: int) -> dict:\n        row = self.rows[idx]\n        sample_dir = self.cache_root / row[\"split\"] / row[\"id\"]\n        input_rgb = _load_rgb(sample_dir / \"input.png\")\n        target = _load_rgb(sample_dir / \"target.png\")\n        mask = _load_gray(sample_dir / \"mask.png\")\n        distance = _load_gray(sample_dir / \"distance.png\")\n        seam_x = 128 + int(row[\"strip\"][\"seam_jitter_px\"])\n        height, width = input_rgb.shape[-2:]\n        boundary = make_boundary_band_mask(height, width, seam_x, 24).squeeze(0)\n        decay = build_decay_mask(height, width, seam_x, int(row[\"strip\"][\"inner_width\"])).squeeze(0)\n        return {\n            \"input\": torch.cat([input_rgb, mask, distance], dim=0),\n            \"target\": target,\n            \"input_rgb\": input_rgb,\n            \"mask\": mask,\n            \"distance\": distance,\n            \"inner_region_mask\": mask,\n            \"boundary_band_mask\": boundary,\n            \"decay_mask\": decay,\n            \"meta\": {\n                \"image_id\": row[\"id\"],\n                \"axis\": row[\"strip\"][\"axis\"],\n                \"side\": row[\"strip\"][\"original_side\"],\n                \"rotation_k\": row[\"strip\"][\"rotation_k\"],\n                \"flip_h\": row[\"strip\"][\"flip_h\"],\n                \"seam_jitter_px\": row[\"strip\"][\"seam_jitter_px\"],\n                \"inner_width\": row[\"strip\"][\"inner_width\"],\n                \"edge_padded_pixels\": row[\"strip\"][\"edge_padded_pixels\"],\n                \"ops\": row[\"corruption\"][\"ops\"],\n                \"scene_tags\": row.get(\"scene_tags\", []),\n                \"split\": row[\"split\"],\n                \"cluster_id\": row.get(\"cluster_id\"),\n                \"seam_x_frac_in_source\": row[\"strip\"][\"seam_x_frac_in_source\"],\n            },\n        }\n",
  "src/models/__init__.py": "\"\"\"Model modules.\"\"\"\n",
  "src/models/blocks.py": "from __future__ import annotations\n\nimport torch\nfrom torch import nn\nimport torch.nn.functional as F\n\n\ndef gaussian_blur_tensor(x: torch.Tensor, sigma: float) -> torch.Tensor:\n    if sigma <= 0:\n        return x\n    radius = max(1, int(round(sigma * 3)))\n    radius = min(radius, max(1, x.shape[-1] // 2 - 1), max(1, x.shape[-2] // 2 - 1))\n    xs = torch.arange(-radius, radius + 1, device=x.device, dtype=x.dtype)\n    kernel = torch.exp(-(xs**2) / (2 * sigma * sigma))\n    kernel = kernel / kernel.sum()\n    kernel_x = kernel.view(1, 1, 1, -1).repeat(x.shape[1], 1, 1, 1)\n    kernel_y = kernel.view(1, 1, -1, 1).repeat(x.shape[1], 1, 1, 1)\n    x = F.conv2d(F.pad(x, (radius, radius, 0, 0), mode=\"reflect\"), kernel_x, groups=x.shape[1])\n    x = F.conv2d(F.pad(x, (0, 0, radius, radius), mode=\"reflect\"), kernel_y, groups=x.shape[1])\n    return x\n\n\nclass ResBlock(nn.Module):\n    def __init__(self, channels: int, groups: int = 8) -> None:\n        super().__init__()\n        self.norm1 = nn.GroupNorm(groups, channels)\n        self.norm2 = nn.GroupNorm(groups, channels)\n        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)\n        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)\n        self.act = nn.SiLU()\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        residual = x\n        x = self.conv1(self.act(self.norm1(x)))\n        x = self.conv2(self.act(self.norm2(x)))\n        return x + residual\n\n\nclass DownBlock(nn.Module):\n    def __init__(self, in_channels: int, out_channels: int, groups: int = 8) -> None:\n        super().__init__()\n        self.proj = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)\n        self.res = ResBlock(out_channels, groups=groups)\n        self.down = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=2, padding=1)\n\n    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:\n        x = self.proj(x)\n        x = self.res(x)\n        skip = x\n        x = self.down(x)\n        return x, skip\n\n\nclass UpBlock(nn.Module):\n    def __init__(self, in_channels: int, skip_channels: int, out_channels: int, groups: int = 8) -> None:\n        super().__init__()\n        self.up = nn.ConvTranspose2d(in_channels, out_channels, kernel_size=2, stride=2)\n        self.proj = nn.Conv2d(out_channels + skip_channels, out_channels, kernel_size=3, padding=1)\n        self.res = ResBlock(out_channels, groups=groups)\n\n    def forward(self, x: torch.Tensor, skip: torch.Tensor) -> torch.Tensor:\n        x = self.up(x)\n        if x.shape[-2:] != skip.shape[-2:]:\n            x = F.interpolate(x, size=skip.shape[-2:], mode=\"bilinear\", align_corners=False)\n        x = torch.cat([x, skip], dim=1)\n        x = self.proj(x)\n        return self.res(x)\n\n\nclass FiLMGenerator(nn.Module):\n    def __init__(self, channels: int) -> None:\n        super().__init__()\n        hidden = max(channels // 2, 16)\n        self.net = nn.Sequential(\n            nn.Linear(channels, hidden),\n            nn.SiLU(),\n            nn.Linear(hidden, channels * 2),\n        )\n\n    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:\n        stats = x.mean(dim=(-2, -1))\n        gamma, beta = self.net(stats).chunk(2, dim=1)\n        return gamma.unsqueeze(-1).unsqueeze(-1), beta.unsqueeze(-1).unsqueeze(-1)\n",
  "src/models/resunet.py": "from __future__ import annotations\n\nimport torch\nfrom torch import nn\n\nfrom src.data.strip_geometry import build_decay_mask\nfrom src.models.blocks import DownBlock, FiLMGenerator, ResBlock, UpBlock, gaussian_blur_tensor\n\n\nclass SeamResUNet(nn.Module):\n    def __init__(\n        self,\n        in_channels: int = 5,\n        out_channels: int = 3,\n        base_channels: int = 32,\n        groups: int = 8,\n        residual_cap: float = 0.3,\n        residual_mode: str = \"full\",\n        low_freq_sigma: float = 6.0,\n    ) -> None:\n        super().__init__()\n        self.in_channels = in_channels\n        self.out_channels = out_channels\n        self.base_channels = base_channels\n        self.residual_cap = residual_cap\n        self.residual_mode = residual_mode\n        self.low_freq_sigma = low_freq_sigma\n\n        self.stem = nn.Conv2d(in_channels, base_channels, kernel_size=3, padding=1)\n        self.enc1 = DownBlock(base_channels, 32, groups=groups)\n        self.enc2 = DownBlock(32, 64, groups=groups)\n        self.enc3 = DownBlock(64, 128, groups=groups)\n        self.enc4 = DownBlock(128, 256, groups=groups)\n\n        self.lf_branch = nn.Sequential(\n            nn.Conv2d(3, 32, kernel_size=3, padding=1),\n            nn.SiLU(),\n            nn.Conv2d(32, 256, kernel_size=3, stride=16, padding=1),\n        )\n        self.bottleneck = nn.Sequential(ResBlock(256, groups=groups), ResBlock(256, groups=groups))\n        self.film = FiLMGenerator(256)\n        self.dec4 = UpBlock(256, 256, 256, groups=groups)\n        self.dec3 = UpBlock(256, 128, 128, groups=groups)\n        self.dec2 = UpBlock(128, 64, 64, groups=groups)\n        self.dec1 = UpBlock(64, 32, 32, groups=groups)\n        self.head = nn.Conv2d(32, out_channels, kernel_size=3, padding=1)\n        nn.init.zeros_(self.head.weight)\n        nn.init.zeros_(self.head.bias)\n        self.residual_scale = nn.Parameter(torch.tensor(0.1))\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        rgb = x[:, :3]\n        seam_x = int((x[:, 3:4, 0, :].argmax(dim=-1).min().item()))\n        stem = self.stem(x)\n        x1, skip1 = self.enc1(stem)\n        x2, skip2 = self.enc2(x1)\n        x3, skip3 = self.enc3(x2)\n        x4, skip4 = self.enc4(x3)\n        lf = gaussian_blur_tensor(rgb, sigma=self.low_freq_sigma)\n        lf = self.lf_branch(lf)\n        bn = self.bottleneck(x4 + lf)\n        gamma, beta = self.film(bn)\n        bn = bn * (1.0 + gamma) + beta\n        dec = self.dec4(bn, skip4)\n        dec = self.dec3(dec, skip3)\n        dec = self.dec2(dec, skip2)\n        dec = self.dec1(dec, skip1)\n        raw = self.head(dec)\n        scale = torch.clamp(self.residual_scale, 0.0, 1.0)\n        residual = torch.tanh(raw) * self.residual_cap * scale\n        if self.residual_mode == \"low_freq_only\":\n            residual = gaussian_blur_tensor(residual, sigma=self.low_freq_sigma)\n        inner_width = x.shape[-1] - seam_x\n        decay = build_decay_mask(x.shape[-2], x.shape[-1], seam_x, inner_width).to(x.device, x.dtype)\n        return residual * decay\n",
  "src/losses/__init__.py": "\"\"\"Loss modules.\"\"\"\n",
  "src/losses/lowfreq.py": "from __future__ import annotations\n\nimport torch\n\nfrom src.models.blocks import gaussian_blur_tensor\n\n\ndef multiscale_lowfreq_loss(pred: torch.Tensor, target: torch.Tensor, inner_mask: torch.Tensor, sigmas: tuple[float, ...]) -> torch.Tensor:\n    total = pred.new_tensor(0.0)\n    denom = inner_mask.sum().clamp_min(1.0)\n    for sigma in sigmas:\n        p = gaussian_blur_tensor(pred, sigma)\n        t = gaussian_blur_tensor(target, sigma)\n        total = total + ((p - t).abs() * inner_mask).sum() / denom\n    return total / max(len(sigmas), 1)\n",
  "src/losses/perceptual.py": "from __future__ import annotations\n\nimport warnings\n\nimport torch\n\nfrom src.models.blocks import gaussian_blur_tensor\n\ntry:\n    import lpips\nexcept Exception:\n    lpips = None\n\n\nclass BoundaryLPIPSLoss:\n    def __init__(self) -> None:\n        if lpips is None:\n            self.model = None\n        else:\n            try:\n                # LPIPS still triggers torchvision UserWarnings (pretrained) on AlexNet; narrow scope.\n                with warnings.catch_warnings():\n                    warnings.simplefilter(\"ignore\", UserWarning)\n                    self.model = lpips.LPIPS(net=\"alex\", pnet_rand=True)\n            except Exception:\n                self.model = None\n\n    def __call__(self, pred: torch.Tensor, target: torch.Tensor, band_mask: torch.Tensor) -> torch.Tensor:\n        if self.model is None:\n            return pred.new_tensor(0.0)\n        device = pred.device\n        if next(self.model.parameters()).device != device:\n            self.model = self.model.to(device)\n        self.model.eval()\n        hf_pred = pred - gaussian_blur_tensor(pred, sigma=4.0)\n        hf_target = target - gaussian_blur_tensor(target, sigma=4.0)\n        hf_pred = (hf_pred * band_mask) * 2.0 - 1.0\n        hf_target = (hf_target * band_mask) * 2.0 - 1.0\n        return self.model(hf_pred, hf_target).mean()\n",
  "src/losses/residual_guard.py": "from __future__ import annotations\n\nimport torch\n\n\ndef residual_smoothness_loss(residual: torch.Tensor) -> torch.Tensor:\n    dx = residual[..., :, 1:] - residual[..., :, :-1]\n    dy = residual[..., 1:, :] - residual[..., :-1, :]\n    return dx.abs().mean() + dy.abs().mean()\n\n\ndef residual_magnitude_loss(residual: torch.Tensor, cap: float = 0.3) -> torch.Tensor:\n    over = (residual.abs() - cap).clamp(min=0.0)\n    return over.mean()\n",
  "src/losses/seam_losses.py": "from __future__ import annotations\n\nimport torch\n\nfrom src.losses.lowfreq import multiscale_lowfreq_loss\nfrom src.losses.perceptual import BoundaryLPIPSLoss\nfrom src.losses.residual_guard import residual_magnitude_loss, residual_smoothness_loss\n\n\ndef charbonnier(diff: torch.Tensor) -> torch.Tensor:\n    return torch.sqrt(diff * diff + 1e-6)\n\n\ndef sobel_gradients(x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:\n    kernel_x = torch.tensor([[1, 0, -1], [2, 0, -2], [1, 0, -1]], dtype=x.dtype, device=x.device).view(1, 1, 3, 3)\n    kernel_y = torch.tensor([[1, 2, 1], [0, 0, 0], [-1, -2, -1]], dtype=x.dtype, device=x.device).view(1, 1, 3, 3)\n    kernels_x = kernel_x.repeat(x.shape[1], 1, 1, 1)\n    kernels_y = kernel_y.repeat(x.shape[1], 1, 1, 1)\n    gx = torch.nn.functional.conv2d(torch.nn.functional.pad(x, (1, 1, 1, 1), mode=\"reflect\"), kernels_x, groups=x.shape[1])\n    gy = torch.nn.functional.conv2d(torch.nn.functional.pad(x, (1, 1, 1, 1), mode=\"reflect\"), kernels_y, groups=x.shape[1])\n    return gx, gy\n\n\nclass SeamLossComputer:\n    def __init__(self) -> None:\n        self.lpips_hf = BoundaryLPIPSLoss()\n        self.lowfreq_sigmas = (2.0, 4.0, 8.0, 16.0, 32.0)\n\n    def __call__(self, pred: torch.Tensor, target: torch.Tensor, input_rgb: torch.Tensor, inner_mask: torch.Tensor, boundary_band: torch.Tensor, residual: torch.Tensor) -> dict[str, torch.Tensor]:\n        outer_mask = 1.0 - inner_mask\n        diff_inner = (pred - target) * inner_mask\n        diff_boundary = (pred - target) * boundary_band\n        l_inner = charbonnier(diff_inner).mean()\n        l_boundary = charbonnier(diff_boundary).mean()\n        l_lowfreq = multiscale_lowfreq_loss(pred, target, inner_mask, self.lowfreq_sigmas)\n        grad_pred = sobel_gradients(pred)\n        grad_target = sobel_gradients(target)\n        l_grad = (grad_pred[0] - grad_target[0]).abs().mean() + (grad_pred[1] - grad_target[1]).abs().mean()\n        l_identity = ((pred - input_rgb).abs() * outer_mask).mean()\n        l_lpips_hf = self.lpips_hf(pred, target, boundary_band)\n        l_smooth = residual_smoothness_loss(residual)\n        l_mag = residual_magnitude_loss(residual)\n        total = (\n            1.00 * l_inner\n            + 2.00 * l_boundary\n            + 1.00 * l_lowfreq\n            + 0.50 * l_grad\n            + 0.20 * l_identity\n            + 0.10 * l_lpips_hf\n            + 0.05 * l_smooth\n            + 0.01 * l_mag\n        )\n        return {\n            \"total\": total,\n            \"l_inner\": l_inner,\n            \"l_boundary\": l_boundary,\n            \"l_lowfreq_ms\": l_lowfreq,\n            \"l_grad\": l_grad,\n            \"l_identity\": l_identity,\n            \"l_lpips_hf\": l_lpips_hf,\n            \"l_residual_smooth\": l_smooth,\n            \"l_residual_magnitude\": l_mag,\n        }\n",
  "src/metrics/__init__.py": "\"\"\"Metric modules.\"\"\"\n",
  "src/metrics/bootstrap.py": "from __future__ import annotations\n\nimport numpy as np\n\n\ndef bootstrap_ci(values: list[float], n_samples: int = 200, alpha: float = 0.05) -> tuple[float, float]:\n    if not values:\n        return (0.0, 0.0)\n    arr = np.asarray(values, dtype=np.float64)\n    means = []\n    rng = np.random.default_rng(42)\n    for _ in range(n_samples):\n        sample = rng.choice(arr, size=arr.shape[0], replace=True)\n        means.append(float(sample.mean()))\n    lo = float(np.quantile(means, alpha / 2))\n    hi = float(np.quantile(means, 1 - alpha / 2))\n    return lo, hi\n",
  "src/metrics/deltae.py": "from __future__ import annotations\n\nimport numpy as np\nfrom skimage.color import deltaE_ciede2000, rgb2lab\n\n\ndef boundary_ciede2000(pred: np.ndarray, target: np.ndarray, mask: np.ndarray) -> float:\n    pred_lab = rgb2lab(pred.clip(0.0, 1.0))\n    target_lab = rgb2lab(target.clip(0.0, 1.0))\n    delta = deltaE_ciede2000(pred_lab, target_lab)\n    valid = mask.squeeze() > 0.5\n    return float(delta[valid].mean()) if np.any(valid) else 0.0\n",
  "src/metrics/lowfreq_metrics.py": "from __future__ import annotations\n\nimport torch\n\nfrom src.models.blocks import gaussian_blur_tensor\n\n\ndef lowfreq_mae(pred: torch.Tensor, target: torch.Tensor, sigma: float = 16.0) -> torch.Tensor:\n    return (gaussian_blur_tensor(pred, sigma) - gaussian_blur_tensor(target, sigma)).abs().mean()\n",
  "src/metrics/reports.py": "from __future__ import annotations\n\nimport csv\nimport json\nfrom pathlib import Path\n\nfrom src.metrics.bootstrap import bootstrap_ci\n\n\ndef write_summary(run_dir: Path, metric_rows: list[dict]) -> None:\n    run_dir.mkdir(parents=True, exist_ok=True)\n    summary = {}\n    for key in metric_rows[0].keys():\n        values = [float(row[key]) for row in metric_rows]\n        summary[key] = {\n            \"mean\": sum(values) / len(values),\n            \"ci95\": bootstrap_ci(values),\n        }\n    (run_dir / \"summary.json\").write_text(json.dumps(summary, indent=2), encoding=\"utf-8\")\n\n\ndef write_bucket_csv(path: Path, rows: list[dict]) -> None:\n    if not rows:\n        return\n    with path.open(\"w\", encoding=\"utf-8\", newline=\"\") as handle:\n        writer = csv.DictWriter(handle, fieldnames=list(rows[0].keys()))\n        writer.writeheader()\n        writer.writerows(rows)\n",
  "src/metrics/seam_metrics.py": "from __future__ import annotations\n\nimport numpy as np\nimport torch\n\nfrom src.metrics.deltae import boundary_ciede2000\nfrom src.metrics.lowfreq_metrics import lowfreq_mae\nfrom src.models.blocks import gaussian_blur_tensor\n\n\ndef _to_numpy(x: torch.Tensor) -> np.ndarray:\n    return x.detach().cpu().permute(0, 2, 3, 1).numpy()\n\n\ndef match_lowfreq(input_rgb: torch.Tensor, target: torch.Tensor, inner_mask: torch.Tensor, sigma: float = 16.0) -> torch.Tensor:\n    correction = gaussian_blur_tensor(target - input_rgb, sigma=sigma)\n    return (input_rgb + correction * inner_mask).clamp(0.0, 1.0)\n\n\ndef _boundary_mae(pred: torch.Tensor, target: torch.Tensor, boundary_band: torch.Tensor) -> float:\n    return float(((pred - target).abs() * boundary_band).mean().item())\n\n\ndef _residual_abs_p99(residual: torch.Tensor, max_elements: int = 4_000_000) -> float:\n    \"\"\"p99 of |residual|; subsample when flat size exceeds torch.quantile limits (large batch × image).\"\"\"\n    v = residual.abs().detach().float().reshape(-1)\n    n = v.numel()\n    if n == 0:\n        return 0.0\n    if n > max_elements:\n        idx = torch.randint(0, n, (max_elements,), device=v.device, dtype=torch.long)\n        v = v[idx]\n    return float(torch.quantile(v, 0.99).item())\n\n\ndef evaluate_batch(pred: torch.Tensor, target: torch.Tensor, input_rgb: torch.Tensor, inner_mask: torch.Tensor, boundary_band: torch.Tensor, residual: torch.Tensor) -> dict[str, float]:\n    pred_np = _to_numpy(pred)[0]\n    target_np = _to_numpy(target)[0]\n    input_np = _to_numpy(input_rgb)[0]\n    boundary_np = _to_numpy(boundary_band)[0][..., :1]\n    baseline_boundary = boundary_ciede2000(input_np, target_np, boundary_np)\n    oracle = match_lowfreq(input_rgb, target, inner_mask, sigma=16.0)\n    oracle_np = _to_numpy(oracle)[0]\n    oracle_boundary = boundary_ciede2000(oracle_np, target_np, boundary_np)\n    pred_boundary = boundary_ciede2000(pred_np, target_np, boundary_np)\n    metrics = {\n        \"boundary_ciede2000\": pred_boundary,\n        \"boundary_mae\": _boundary_mae(pred, target, boundary_band),\n        \"inner_mae\": float(((pred - target).abs() * inner_mask).mean().item()),\n        \"outer_identity_error\": float(((pred - input_rgb).abs() * (1.0 - inner_mask)).max().item()),\n        \"lowfreq_mae_sigma16\": float(lowfreq_mae(pred, target, sigma=16.0).item()),\n        \"residual_magnitude_p99\": _residual_abs_p99(residual),\n        \"baseline_boundary_ciede2000\": baseline_boundary,\n        \"baseline_boundary_mae\": _boundary_mae(input_rgb, target, boundary_band),\n        \"oracle_boundary_ciede2000\": oracle_boundary,\n        \"relative_improvement\": float((baseline_boundary - pred_boundary) / (baseline_boundary - oracle_boundary + 1e-8)),\n    }\n    return metrics\n",
  "src/train/__init__.py": "\"\"\"Training modules.\"\"\"\n",
  "src/train/checkpoint.py": "from __future__ import annotations\n\nimport json\nimport os\nimport random\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\n\n\ndef save_checkpoint(path: Path, payload: dict) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    torch.save(payload, path)\n\n\ndef load_checkpoint(path: Path, map_location: str = \"cpu\") -> dict:\n    return torch.load(path, map_location=map_location)\n\n\ndef capture_rng_state() -> dict:\n    state = {\n        \"torch\": torch.get_rng_state(),\n        \"numpy\": np.random.get_state(),\n        \"python\": random.getstate(),\n    }\n    if torch.cuda.is_available():\n        state[\"cuda\"] = torch.cuda.get_rng_state_all()\n    return state\n\n\ndef restore_rng_state(state: dict) -> None:\n    torch.set_rng_state(state[\"torch\"])\n    np.random.set_state(state[\"numpy\"])\n    random.setstate(state[\"python\"])\n    if torch.cuda.is_available() and \"cuda\" in state:\n        torch.cuda.set_rng_state_all(state[\"cuda\"])\n\n\ndef config_hash(config: dict) -> str:\n    return str(abs(hash(json.dumps(config, sort_keys=True))))\n\n\ndef git_hash() -> str:\n    return os.environ.get(\"GIT_HASH\", \"nogit\")\n\n\ndef save_training_checkpoint(\n    path: Path,\n    *,\n    model: torch.nn.Module,\n    ema_state: dict,\n    optimizer: torch.optim.Optimizer,\n    scheduler: object | None,\n    scaler: object | None,\n    epoch: int,\n    config: dict,\n    metrics: dict,\n) -> None:\n    payload = {\n        \"model\": model.state_dict(),\n        \"ema\": ema_state,\n        \"optimizer\": optimizer.state_dict(),\n        \"scheduler\": scheduler.state_dict() if scheduler is not None else None,\n        \"scaler\": scaler.state_dict() if scaler is not None else None,\n        \"epoch\": epoch,\n        \"rng_state\": capture_rng_state(),\n        \"config_hash\": config_hash(config),\n        \"git_hash\": git_hash(),\n        \"metrics\": metrics,\n    }\n    save_checkpoint(path, payload)\n",
  "src/train/ema.py": "from __future__ import annotations\n\nimport copy\n\nimport torch\n\n\nclass EMA:\n    def __init__(self, model: torch.nn.Module, decay: float = 0.999) -> None:\n        self.decay = decay\n        self.model = copy.deepcopy(model).eval()\n        for param in self.model.parameters():\n            param.requires_grad_(False)\n\n    @torch.no_grad()\n    def update(self, model: torch.nn.Module) -> None:\n        ema_state = self.model.state_dict()\n        for key, value in model.state_dict().items():\n            ema_state[key].mul_(self.decay).add_(value.detach(), alpha=1.0 - self.decay)\n\n    def state_dict(self) -> dict:\n        return self.model.state_dict()\n\n    def load_state_dict(self, state: dict) -> None:\n        self.model.load_state_dict(state)\n",
  "src/train/scheduler.py": "from __future__ import annotations\n\nimport math\n\nfrom torch.optim import Optimizer\nfrom torch.optim.lr_scheduler import LambdaLR\n\n\ndef cosine_with_warmup(optimizer: Optimizer, warmup_steps: int, total_steps: int, min_lr_scale: float = 0.005) -> LambdaLR:\n    def fn(step: int) -> float:\n        if step < warmup_steps:\n            return max(step / max(warmup_steps, 1), 1e-8)\n        progress = (step - warmup_steps) / max(total_steps - warmup_steps, 1)\n        cosine = 0.5 * (1.0 + math.cos(math.pi * progress))\n        return min_lr_scale + (1.0 - min_lr_scale) * cosine\n\n    return LambdaLR(optimizer, lr_lambda=fn)\n",
  "src/train/train_loop.py": "from __future__ import annotations\n\nimport contextlib\nimport json\nimport sys\nimport time\nfrom dataclasses import dataclass\nfrom typing import Any\n\nimport torch\nfrom torch.amp import GradScaler, autocast\nfrom torch.utils.data import DataLoader\nfrom tqdm.auto import tqdm\n\nfrom src.losses.seam_losses import SeamLossComputer\nfrom src.metrics.seam_metrics import evaluate_batch\nfrom src.train.ema import EMA\n\n\n@dataclass\nclass EpochResult:\n    losses: dict[str, float]\n    metrics: dict[str, float]\n    per_sample_metrics: list[dict[str, float]]\n\n\ndef _move(batch: dict, device: torch.device) -> dict:\n    out = {}\n    for key, value in batch.items():\n        if isinstance(value, torch.Tensor):\n            out[key] = value.to(device)\n        else:\n            out[key] = value\n    return out\n\n\ndef run_epoch(\n    model: torch.nn.Module,\n    loader: DataLoader,\n    optimizer: torch.optim.Optimizer | None,\n    device: torch.device,\n    ema: EMA | None = None,\n    scaler: GradScaler | None = None,\n    scheduler: torch.optim.lr_scheduler._LRScheduler | None = None,\n    use_amp: bool = False,\n    desc: str | None = None,\n    tb_writer: Any = None,\n    tb_prefix: str = \"train\",\n    tb_global_step: int = 0,\n    tb_log_interval: int = 1,\n    console_log_interval: int = 0,\n    loss_computer: SeamLossComputer | None = None,\n    wall_t0: float | None = None,\n    current_epoch: int | None = None,\n    total_epochs: int | None = None,\n) -> tuple[EpochResult, int]:\n    if loss_computer is None:\n        loss_computer = SeamLossComputer()\n    train_mode = optimizer is not None\n    model.train(train_mode)\n    agg_losses: dict[str, float] = {}\n    agg_metrics: dict[str, float] = {}\n    per_sample_metrics: list[dict[str, float]] = []\n    steps = 0\n    use_tqdm = sys.stderr.isatty()\n    n_batches = len(loader) if hasattr(loader, \"__len__\") else None\n    progress = tqdm(\n        loader,\n        desc=desc or (\"train\" if train_mode else \"eval\"),\n        dynamic_ncols=True,\n        leave=False,\n        disable=not use_tqdm,\n    )\n    t_epoch_start = time.monotonic()\n    if not train_mode and console_log_interval > 0 and n_batches is not None:\n        vpre: dict[str, Any] = {\n            \"event\": \"val_iter_begin\",\n            \"desc\": desc,\n            \"batches_in_epoch\": n_batches,\n        }\n        if wall_t0 is not None:\n            vpre[\"sec_since_train_start\"] = int(time.perf_counter() - wall_t0)\n        print(json.dumps(vpre, ensure_ascii=False), flush=True)\n    if train_mode and console_log_interval > 0 and n_batches is not None:\n        tpre: dict[str, Any] = {\n            \"event\": \"train_iter_begin\",\n            \"desc\": desc,\n            \"batches_in_epoch\": n_batches,\n            \"note\": \"first DataLoader batch + 1st forward/backward can take minutes; next: train_step\",\n        }\n        if wall_t0 is not None:\n            tpre[\"sec_since_train_start\"] = int(time.perf_counter() - wall_t0)\n        print(json.dumps(tpre, ensure_ascii=False), flush=True)\n    _amp_device_types = frozenset({\"cuda\", \"cpu\", \"mps\", \"hpu\", \"xpu\", \"mtia\"})\n    for batch in progress:\n        batch = _move(batch, device)\n        inputs = batch[\"input\"]\n        input_rgb = batch[\"input_rgb\"]\n        target = batch[\"target\"]\n        inner_mask = batch[\"inner_region_mask\"]\n        boundary = batch[\"boundary_band_mask\"]\n        ad = inputs.device.type\n        if ad in _amp_device_types:\n            amp_ctx = autocast(device_type=ad, enabled=use_amp)\n        else:\n            amp_ctx = contextlib.nullcontext()\n        with amp_ctx:\n            residual = model(inputs)\n            pred = (input_rgb + residual).clamp(0.0, 1.0)\n            pred[:, :, :, :128] = input_rgb[:, :, :, :128]\n            losses = loss_computer(pred, target, input_rgb, inner_mask, boundary, residual)\n        if train_mode:\n            assert optimizer is not None\n            optimizer.zero_grad(set_to_none=True)\n            did_optim_step = False\n            if scaler is not None and use_amp:\n                scaler.scale(losses[\"total\"]).backward()\n                scaler.unscale_(optimizer)\n                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n                _prev = optimizer._step_count\n                scaler.step(optimizer)\n                scaler.update()\n                did_optim_step = optimizer._step_count > _prev\n            else:\n                losses[\"total\"].backward()\n                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n                _prev = optimizer._step_count\n                optimizer.step()\n                did_optim_step = optimizer._step_count > _prev\n            if scheduler is not None and did_optim_step:\n                scheduler.step()\n            if ema is not None:\n                ema.update(model)\n        metrics = evaluate_batch(pred.detach(), target, input_rgb, inner_mask, boundary, residual.detach())\n        per_sample_metrics.append(metrics)\n        for key, value in losses.items():\n            agg_losses[key] = agg_losses.get(key, 0.0) + float(value.detach().item())\n        for key, value in metrics.items():\n            agg_metrics[key] = agg_metrics.get(key, 0.0) + float(value)\n        steps += 1\n        if tb_writer is not None and train_mode and tb_log_interval > 0:\n            gs = tb_global_step + steps\n            if (\n                steps == 1\n                or steps % tb_log_interval == 0\n                or (n_batches is not None and steps == n_batches)\n            ):\n                w = float(agg_losses.get(\"total\", 0.0) / steps)\n                tb_writer.add_scalar(f\"{tb_prefix}/loss/total\", w, gs)\n                for lk, lv in agg_losses.items():\n                    if lk == \"total\":\n                        continue\n                    tb_writer.add_scalar(f\"{tb_prefix}/loss/{lk}\", float(lv) / steps, gs)\n                for mk, mv in agg_metrics.items():\n                    tb_writer.add_scalar(f\"{tb_prefix}/metric/{mk}\", float(mv) / steps, gs)\n                if scheduler is not None and optimizer is not None:\n                    lr = optimizer.param_groups[0].get(\"lr\", 0.0)\n                    tb_writer.add_scalar(f\"{tb_prefix}/lr\", float(lr), gs)\n                tb_writer.flush()\n        progress.set_postfix(\n            loss=f\"{agg_losses.get('total', 0.0) / steps:.4f}\",\n            b_ciede=f\"{agg_metrics.get('boundary_ciede2000', 0.0) / steps:.3f}\",\n            rel=f\"{agg_metrics.get('relative_improvement', 0.0) / steps:.3f}\",\n        )\n        if console_log_interval > 0:\n            if steps == 1 or steps % console_log_interval == 0 or (n_batches is not None and steps == n_batches):\n                gs = tb_global_step + steps\n                elapsed_epoch = time.monotonic() - t_epoch_start\n                eta_epoch: float | None = None\n                if n_batches and steps > 0 and steps < n_batches:\n                    sec_per = elapsed_epoch / steps\n                    eta_epoch = sec_per * (n_batches - steps)\n                out: dict[str, Any] = {\n                    \"event\": \"train_step\" if train_mode else \"val_step\",\n                    \"desc\": desc,\n                    \"epoch\": current_epoch,\n                    \"of_epochs\": total_epochs,\n                    \"global_step\": gs,\n                    \"step_in_epoch\": steps,\n                    \"batches_in_epoch\": n_batches,\n                    \"progress_in_epoch\": round(100.0 * steps / n_batches, 1) if n_batches and n_batches > 0 else None,\n                    \"loss_total\": round(agg_losses.get(\"total\", 0.0) / steps, 6),\n                    \"b_ciede\": round(agg_metrics.get(\"boundary_ciede2000\", 0.0) / steps, 4),\n                }\n                if wall_t0 is not None:\n                    out[\"sec_since_train_start\"] = int(time.perf_counter() - wall_t0)\n                if eta_epoch is not None:\n                    out[\"eta_sec_this_epoch\"] = int(eta_epoch)\n                print(json.dumps(out, ensure_ascii=False), flush=True)\n    progress.close()\n    if steps == 0:\n        return EpochResult({}, {}, []), tb_global_step\n    end_step = tb_global_step + steps if train_mode else tb_global_step\n    return (\n        EpochResult(\n            losses={key: value / steps for key, value in agg_losses.items()},\n            metrics={key: value / steps for key, value in agg_metrics.items()},\n            per_sample_metrics=per_sample_metrics,\n        ),\n        end_step,\n    )\n",
  "src/utils/__init__.py": "\"\"\"Utility modules.\"\"\"\n",
  "src/utils/device.py": "from __future__ import annotations\n\nimport torch\n\n\ndef pick_device() -> torch.device:\n    if torch.cuda.is_available():\n        return torch.device(\"cuda\")\n    if getattr(torch.backends, \"mps\", None) is not None and torch.backends.mps.is_available():\n        return torch.device(\"mps\")\n    return torch.device(\"cpu\")\n\n\ndef amp_enabled(device: torch.device, precision: str) -> bool:\n    return device.type == \"cuda\" and precision in {\"fp16\", \"bf16\"}\n",
  "src/utils/image_io.py": "from __future__ import annotations\n\nimport hashlib\nfrom io import BytesIO\nfrom pathlib import Path\nfrom typing import Iterable\n\nimport numpy as np\nfrom PIL import Image, ImageCms, ImageOps\n\n\nIMAGE_EXTENSIONS = {\".jpg\", \".jpeg\", \".png\"}\n\n\ndef iter_image_files(root: Path) -> Iterable[Path]:\n    for path in sorted(root.iterdir()):\n        if path.suffix.lower() in IMAGE_EXTENSIONS:\n            yield path\n\n\ndef open_rgb_image(path: Path) -> Image.Image:\n    image = Image.open(path)\n    image = ImageOps.exif_transpose(image)\n    if image.mode == \"RGBA\":\n        raise ValueError(\"rgba_not_allowed\")\n    if \"icc_profile\" in image.info:\n        try:\n            src = ImageCms.ImageCmsProfile(BytesIO(image.info[\"icc_profile\"]))\n            dst = ImageCms.createProfile(\"sRGB\")\n            image = ImageCms.profileToProfile(image, src, dst, outputMode=image.mode)\n        except Exception:\n            pass\n    if image.mode in {\"L\", \"P\", \"CMYK\"}:\n        image = image.convert(\"RGB\")\n    elif image.mode != \"RGB\":\n        image = image.convert(\"RGB\")\n    return image\n\n\ndef center_crop_square(image: Image.Image) -> Image.Image:\n    w, h = image.size\n    side = min(w, h)\n    left = (w - side) // 2\n    top = (h - side) // 2\n    return image.crop((left, top, left + side, top + side))\n\n\ndef resize_square(image: Image.Image, size: int = 1024) -> Image.Image:\n    return image.resize((size, size), Image.Resampling.LANCZOS)\n\n\ndef save_png(image: Image.Image, path: Path) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    image.save(path, format=\"PNG\")\n\n\ndef pil_to_numpy(image: Image.Image) -> np.ndarray:\n    return np.asarray(image.convert(\"RGB\"), dtype=np.uint8)\n\n\ndef sha256_file(path: Path) -> str:\n    h = hashlib.sha256()\n    with path.open(\"rb\") as handle:\n        for chunk in iter(lambda: handle.read(1024 * 1024), b\"\"):\n            h.update(chunk)\n    return h.hexdigest()\n",
  "src/utils/phash.py": "from __future__ import annotations\n\nfrom pathlib import Path\n\nimport numpy as np\nfrom PIL import Image\nfrom scipy.fftpack import dct\n\n\ndef _phash_from_image(image: Image.Image, hash_size: int = 8, highfreq_factor: int = 4) -> str:\n    img_size = hash_size * highfreq_factor\n    image = image.convert(\"L\").resize((img_size, img_size), Image.Resampling.LANCZOS)\n    pixels = np.asarray(image, dtype=np.float32)\n    dct_rows = dct(pixels, axis=0, norm=\"ortho\")\n    dct_full = dct(dct_rows, axis=1, norm=\"ortho\")\n    lowfreq = dct_full[:hash_size, :hash_size]\n    med = np.median(lowfreq[1:, 1:])\n    bits = lowfreq > med\n    flat = \"\".join(\"1\" if bit else \"0\" for bit in bits.flatten())\n    return f\"{int(flat, 2):0{hash_size * hash_size // 4}x}\"\n\n\ndef compute_phash64(path: Path) -> str:\n    with Image.open(path) as image:\n        return _phash_from_image(image, hash_size=8)\n\n\ndef hamming_distance(hex_a: str, hex_b: str) -> int:\n    return bin(int(hex_a, 16) ^ int(hex_b, 16)).count(\"1\")\n",
  "src/utils/seed.py": "from __future__ import annotations\n\nimport random\n\nimport numpy as np\nimport torch\n\n\ndef seed_everything(seed: int) -> None:\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n\n\ndef worker_init_fn(worker_id: int) -> None:\n    base = torch.initial_seed() % (2**32)\n    np.random.seed(base + worker_id)\n    random.seed(base + worker_id)\n",
  "scripts/train_resunet.py": "from __future__ import annotations\n\nimport sys\n\n# Before heavy imports: Colab is often quiet for 30–90s while torch loads.\nif __name__ == \"__main__\":\n    print(\"train_resunet: loading PyTorch and project code (this can take 30–90s)…\", flush=True)\n\nimport argparse\nimport json\nimport os\nimport time\nfrom pathlib import Path\nfrom typing import Any\n\nimport torch\nimport yaml\nfrom torch.amp import GradScaler\nfrom torch.utils.data import DataLoader\nfrom tqdm.auto import tqdm\n\nfrom src.data.cached_strip_dataset import CachedStripDataset\nfrom src.data.synthetic_strip_dataset import SyntheticStripDataset, collate_strip_batch\nfrom src.losses.seam_losses import SeamLossComputer\nfrom src.models.resunet import SeamResUNet\nfrom src.train.checkpoint import load_checkpoint, restore_rng_state, save_training_checkpoint\nfrom src.train.ema import EMA\nfrom src.train.scheduler import cosine_with_warmup\nfrom src.train.train_loop import run_epoch\nfrom src.utils.device import amp_enabled, pick_device\nfrom src.utils.seed import seed_everything, worker_init_fn\n\n\ndef main() -> None:\n    # Before TensorFlow (pulled in by torch.utils.tensorboard) loads — fewer cuDNN/oneDNN stderr lines.\n    os.environ.setdefault(\"TF_CPP_MIN_LOG_LEVEL\", \"2\")\n    os.environ.setdefault(\"TF_ENABLE_ONEDNN_OPTS\", \"0\")\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\"--config\", default=\"configs/train_synth_v1.yaml\")\n    parser.add_argument(\"--resume\", default=None)\n    parser.add_argument(\"--max-epochs\", type=int, default=None)\n    args = parser.parse_args()\n    cfg = yaml.safe_load(Path(args.config).read_text(encoding=\"utf-8\"))\n    seed_everything(cfg[\"seed\"])\n    device = pick_device()\n    num_epochs = args.max_epochs or cfg[\"train\"][\"num_epochs\"]\n    print(\n        json.dumps({\"device\": str(device), \"epochs\": num_epochs, \"batch_size\": cfg[\"train\"][\"batch_size\"]}, ensure_ascii=False),\n        flush=True,\n    )\n    cache_root = Path(cfg[\"dataset\"].get(\"cache_root\", \"outputs/strip_cache\"))\n    train_cache_manifest = Path(cfg[\"dataset\"].get(\"train_cache_manifest\", \"manifests/strip_train_cache.jsonl\"))\n    val_cache_manifest = Path(cfg[\"dataset\"].get(\"val_cache_manifest\", \"manifests/strip_val_cache.jsonl\"))\n    if cache_root.exists() and train_cache_manifest.exists() and val_cache_manifest.exists():\n        train_ds = CachedStripDataset(train_cache_manifest, cache_root)\n        val_ds = CachedStripDataset(val_cache_manifest, cache_root)\n    else:\n        train_ds = SyntheticStripDataset(Path(cfg[\"dataset\"][\"source_manifest\"]), split=\"train\", strips_per_image=cfg[\"dataset\"][\"strips_per_image\"])\n        val_ds = SyntheticStripDataset(Path(cfg[\"dataset\"][\"source_manifest\"]), split=\"val\", strips_per_image=max(1, cfg[\"dataset\"][\"strips_per_image\"] // 4))\n    train_workers = int(cfg[\"train\"][\"num_workers\"])\n    val_workers = min(max(train_workers // 2, 0), 8)\n    common_loader = {\n        \"pin_memory\": device.type == \"cuda\",\n        \"collate_fn\": collate_strip_batch,\n    }\n    if train_workers > 0:\n        common_loader[\"persistent_workers\"] = True\n        common_loader[\"prefetch_factor\"] = 2\n    train_loader = DataLoader(train_ds, batch_size=cfg[\"train\"][\"batch_size\"], shuffle=True, num_workers=train_workers, worker_init_fn=worker_init_fn, **common_loader)\n    val_loader = DataLoader(val_ds, batch_size=max(1, cfg[\"train\"][\"batch_size\"] // 2), shuffle=False, num_workers=val_workers, worker_init_fn=worker_init_fn if val_workers > 0 else None, **common_loader)\n    model = SeamResUNet(residual_mode=cfg[\"model\"][\"residual_mode\"], low_freq_sigma=cfg[\"model\"][\"low_freq_sigma\"]).to(device)\n    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg[\"train\"][\"lr\"], weight_decay=cfg[\"train\"][\"weight_decay\"], betas=tuple(cfg[\"train\"][\"betas\"]))\n    ema = EMA(model, decay=cfg[\"ema\"][\"decay\"])\n    _amp = amp_enabled(device, cfg[\"train\"][\"precision\"])\n    scaler = (\n        GradScaler(\"cuda\", enabled=_amp) if device.type == \"cuda\" else GradScaler(\"cpu\", enabled=False)\n    )\n    total_steps = max(len(train_loader) * num_epochs, 1)\n    scheduler = cosine_with_warmup(optimizer, warmup_steps=cfg[\"scheduler\"][\"warmup_steps\"], total_steps=total_steps)\n    start_epoch = 0\n    best = {\n        \"boundary_ciede2000\": float(\"inf\"),\n        \"boundary_mae\": float(\"inf\"),\n        \"outer_identity_error\": float(\"inf\"),\n        \"relative_improvement\": float(\"-inf\"),\n    }\n    if args.resume:\n        state = load_checkpoint(Path(args.resume), map_location=device.type)\n        model.load_state_dict(state[\"model\"])\n        ema.load_state_dict(state[\"ema\"])\n        optimizer.load_state_dict(state[\"optimizer\"])\n        if state.get(\"scheduler\") is not None:\n            scheduler.load_state_dict(state[\"scheduler\"])\n        if state.get(\"scaler\") is not None:\n            scaler.load_state_dict(state[\"scaler\"])\n        restore_rng_state(state[\"rng_state\"])\n        start_epoch = int(state[\"epoch\"]) + 1\n        print(json.dumps({\"event\": \"resumed\", \"start_epoch\": start_epoch}, ensure_ascii=False), flush=True)\n    log_cfg: dict = cfg.get(\"logging\") or {}\n    console_iv = int(log_cfg.get(\"console_log_interval\", 25))\n    print(\n        json.dumps(\n            {\n                \"event\": \"train_start\",\n                \"train_samples\": len(train_ds),\n                \"val_samples\": len(val_ds),\n                \"batches_per_epoch\": len(train_loader),\n                \"val_batches_per_epoch\": len(val_loader),\n                \"num_epochs\": num_epochs,\n                \"start_epoch\": start_epoch,\n                \"end_epoch_excl\": num_epochs,\n                \"console_log_interval\": console_iv,\n                \"note\": \"In Colab subprocess tqdm is off; use these JSON lines + TensorBoard.\",\n            },\n            ensure_ascii=False,\n        ),\n        flush=True,\n    )\n    tb_on = bool(log_cfg.get(\"tensorboard\", True))\n    log_dir: Path = Path(log_cfg.get(\"log_dir\", \"outputs/logs/tensorboard\"))\n    log_interval = int(log_cfg.get(\"log_interval\", 20))\n    tb_writer: Any = None\n    if tb_on:\n        try:\n            from torch.utils.tensorboard import SummaryWriter\n\n            log_dir.mkdir(parents=True, exist_ok=True)\n            tb_writer = SummaryWriter(str(log_dir))\n            # So TensorBoard has an event file before the first log_interval training steps (otherwise \"No dashboards\").\n            tb_writer.add_scalar(\"meta/run_started\", 1.0, 0)\n            tb_writer.flush()\n            print(json.dumps({\"tensorboard_logdir\": str(log_dir.resolve())}, ensure_ascii=False), flush=True)\n        except Exception as e:  # noqa: BLE001\n            print(json.dumps({\"tensorboard\": \"disabled\", \"reason\": str(e)}, ensure_ascii=False), flush=True)\n    global_step = 0\n    # One LPIPS / SeamLossComputer for all epochs (avoids re-loading alex.net weights every epoch).\n    loss_computer = SeamLossComputer()\n    epoch_use_tqdm = sys.stderr.isatty()\n    epoch_bar = tqdm(\n        range(start_epoch, num_epochs),\n        desc=\"epochs\",\n        dynamic_ncols=True,\n        disable=not epoch_use_tqdm,\n    )\n    wall_t0 = time.perf_counter()\n    for epoch in epoch_bar:\n        epoch_start = time.time()\n        print(\n            json.dumps(\n                {\n                    \"event\": \"epoch_begin\",\n                    \"epoch\": epoch + 1,\n                    \"of\": num_epochs,\n                    \"sec_since_train_start\": int(time.perf_counter() - wall_t0),\n                },\n                ensure_ascii=False,\n            ),\n            flush=True,\n        )\n        train_result, global_step = run_epoch(\n            model,\n            train_loader,\n            optimizer,\n            device,\n            ema=ema,\n            scaler=scaler,\n            scheduler=scheduler,\n            use_amp=scaler.is_enabled(),\n            desc=f\"train e{epoch+1}/{num_epochs}\",\n            tb_writer=tb_writer,\n            tb_prefix=\"train\",\n            tb_global_step=global_step,\n            tb_log_interval=max(1, log_interval),\n            console_log_interval=console_iv,\n            loss_computer=loss_computer,\n            wall_t0=wall_t0,\n            current_epoch=epoch + 1,\n            total_epochs=num_epochs,\n        )\n        # Flush before val: no train_step/val_step lines until a batch finishes; Colab may look \"stuck\".\n        print(\n            json.dumps(\n                {\n                    \"event\": \"val_begin\",\n                    \"epoch\": epoch + 1,\n                    \"of\": num_epochs,\n                    \"val_batches\": len(val_loader),\n                    \"sec_since_train_start\": int(time.perf_counter() - wall_t0),\n                },\n                ensure_ascii=False,\n            ),\n            flush=True,\n        )\n        val_result, _ = run_epoch(\n            ema.model,\n            val_loader,\n            None,\n            device,\n            use_amp=False,\n            desc=f\"val e{epoch+1}/{num_epochs}\",\n            tb_global_step=global_step,\n            console_log_interval=console_iv,\n            loss_computer=loss_computer,\n            wall_t0=wall_t0,\n            current_epoch=epoch + 1,\n            total_epochs=num_epochs,\n        )\n        if tb_writer is not None:\n            for k, v in val_result.losses.items():\n                tb_writer.add_scalar(f\"val/loss_{k}\", v, global_step)\n            for k, v in val_result.metrics.items():\n                tb_writer.add_scalar(f\"val/metric_{k}\", v, global_step)\n            tb_writer.add_scalar(\"epoch\", float(epoch), global_step)\n            tb_writer.flush()\n        metrics = {\"train\": train_result.metrics, \"val\": val_result.metrics}\n        save_training_checkpoint(Path(\"outputs/checkpoints/last.pt\"), model=model, ema_state=ema.state_dict(), optimizer=optimizer, scheduler=scheduler, scaler=scaler, epoch=epoch, config=cfg, metrics=metrics)\n        if val_result.metrics.get(\"boundary_ciede2000\", float(\"inf\")) < best[\"boundary_ciede2000\"]:\n            best[\"boundary_ciede2000\"] = val_result.metrics[\"boundary_ciede2000\"]\n            save_training_checkpoint(Path(\"outputs/checkpoints/best_boundary_ciede2000.pt\"), model=model, ema_state=ema.state_dict(), optimizer=optimizer, scheduler=scheduler, scaler=scaler, epoch=epoch, config=cfg, metrics=metrics)\n        if val_result.metrics.get(\"boundary_mae\", float(\"inf\")) < best[\"boundary_mae\"]:\n            best[\"boundary_mae\"] = val_result.metrics[\"boundary_mae\"]\n            save_training_checkpoint(Path(\"outputs/checkpoints/best_boundary_mae.pt\"), model=model, ema_state=ema.state_dict(), optimizer=optimizer, scheduler=scheduler, scaler=scaler, epoch=epoch, config=cfg, metrics=metrics)\n        if val_result.metrics.get(\"outer_identity_error\", float(\"inf\")) < best[\"outer_identity_error\"]:\n            best[\"outer_identity_error\"] = val_result.metrics[\"outer_identity_error\"]\n            save_training_checkpoint(Path(\"outputs/checkpoints/best_outer_identity.pt\"), model=model, ema_state=ema.state_dict(), optimizer=optimizer, scheduler=scheduler, scaler=scaler, epoch=epoch, config=cfg, metrics=metrics)\n        if val_result.metrics.get(\"relative_improvement\", float(\"-inf\")) > best[\"relative_improvement\"]:\n            best[\"relative_improvement\"] = val_result.metrics[\"relative_improvement\"]\n            save_training_checkpoint(Path(\"outputs/checkpoints/best_relative_improvement.pt\"), model=model, ema_state=ema.state_dict(), optimizer=optimizer, scheduler=scheduler, scaler=scaler, epoch=epoch, config=cfg, metrics=metrics)\n        epoch_bar.set_postfix(\n            sec=f\"{time.time()-epoch_start:.1f}\",\n            val_b=f\"{val_result.metrics.get('boundary_ciede2000', 0.0):.3f}\",\n            rel=f\"{val_result.metrics.get('relative_improvement', 0.0):.3f}\",\n        )\n        rem_epochs = num_epochs - (epoch + 1)\n        # Rough ETA for the rest of training: (this epoch wall time) * epochs_left (ignores that epochs may get slower).\n        est_remaining_sec = int((time.time() - epoch_start) * rem_epochs) if rem_epochs > 0 else None\n        print(\n            json.dumps(\n                {\n                    \"event\": \"epoch_end\",\n                    \"epoch\": epoch + 1,\n                    \"of\": num_epochs,\n                    \"epochs_left\": rem_epochs,\n                    \"sec_epoch_wall\": round(time.time() - epoch_start, 1),\n                    \"sec_since_train_start\": int(time.perf_counter() - wall_t0),\n                    \"eta_sec_full_run_rough\": est_remaining_sec,\n                    \"train_loss\": train_result.losses,\n                    \"val_metrics\": val_result.metrics,\n                },\n                ensure_ascii=False,\n            ),\n            flush=True,\n        )\n    if tb_writer is not None:\n        tb_writer.close()\n\n\nif __name__ == \"__main__\":\n    main()\n",
  "scripts/run_eval.py": "from __future__ import annotations\n\nimport argparse\nimport json\nfrom datetime import datetime\nfrom pathlib import Path\n\nimport torch\nimport yaml\nfrom PIL import Image\nfrom torch.utils.data import DataLoader\nfrom tqdm.auto import tqdm\n\nfrom src.data.cached_strip_dataset import CachedStripDataset\nfrom src.data.synthetic_strip_dataset import SyntheticStripDataset, collate_strip_batch\nfrom src.metrics.bootstrap import bootstrap_ci\nfrom src.metrics.reports import write_bucket_csv, write_summary\nfrom src.models.resunet import SeamResUNet\nfrom src.train.checkpoint import load_checkpoint\nfrom src.train.train_loop import run_epoch\nfrom src.utils.device import pick_device\n\n\ndef _gate_status(metrics: dict[str, float]) -> dict:\n    return {\n        \"boundary_ciede2000_mean\": metrics.get(\"boundary_ciede2000\", 1.0) < 0.7 * max(metrics.get(\"baseline_boundary_ciede2000\", 1.0), 1e-8),\n        \"boundary_ciede2000_p95\": metrics.get(\"boundary_ciede2000\", 1.0) < 0.8 * max(metrics.get(\"baseline_boundary_ciede2000\", 1.0), 1e-8),\n        \"outer_identity_mae\": metrics.get(\"outer_identity_error\", 1.0) < 1e-6,\n        \"lpips_hf_delta_mean\": True,\n        \"residual_magnitude_p99\": metrics.get(\"residual_magnitude_p99\", 1.0) < 0.3,\n        \"worst_bucket_relative_improvement\": metrics.get(\"relative_improvement\", 0.0) > 0.2,\n    }\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\"--config\", default=\"configs/eval_v1.yaml\")\n    args = parser.parse_args()\n    cfg = yaml.safe_load(Path(args.config).read_text(encoding=\"utf-8\"))\n    device = pick_device()\n    ckpt = load_checkpoint(Path(cfg[\"checkpoint\"]), map_location=device.type)\n    model = SeamResUNet().to(device)\n    model.load_state_dict(ckpt[\"ema\"])\n    cache_root = Path(cfg.get(\"cache_root\", \"outputs/strip_cache\"))\n    val_cache_manifest = Path(cfg.get(\"val_cache_manifest\", \"manifests/strip_val_cache.jsonl\"))\n    if cache_root.exists() and val_cache_manifest.exists():\n        dataset = CachedStripDataset(val_cache_manifest, cache_root)\n    else:\n        dataset = SyntheticStripDataset(Path(\"manifests/input_raw_manifest.jsonl\"), split=\"val\", strips_per_image=1)\n    loader = DataLoader(dataset, batch_size=1, shuffle=False, collate_fn=collate_strip_batch)\n    print(json.dumps({\"device\": str(device), \"val_samples\": len(dataset)}, ensure_ascii=False))\n    result, _ = run_epoch(model, loader, None, device, desc=\"eval\")\n    run_dir = Path(cfg[\"report_root\"]) / f\"run_{datetime.now().strftime('%Y%m%d_%H%M%S')}\"\n    run_dir.mkdir(parents=True, exist_ok=True)\n    metric_rows = result.per_sample_metrics or [result.metrics]\n    write_summary(run_dir, metric_rows)\n    write_bucket_csv(run_dir / \"metrics_by_bucket.csv\", metric_rows)\n    gates = _gate_status(result.metrics)\n    (run_dir / \"gates.txt\").write_text(\"\\n\".join(f\"{key}: {'PASS' if value else 'FAIL'}\" for key, value in gates.items()), encoding=\"utf-8\")\n    summary_json = json.loads((run_dir / \"summary.json\").read_text(encoding=\"utf-8\"))\n    for key, values in summary_json.items():\n        series = [row[key] for row in metric_rows if key in row]\n        summary_json[key][\"ci95\"] = bootstrap_ci(series, n_samples=cfg.get(\"bootstrap_samples\", 200))\n    (run_dir / \"summary.json\").write_text(json.dumps(summary_json, indent=2), encoding=\"utf-8\")\n    for bucket in (\"best_10\", \"median_10\", \"worst_10\", \"grids\"):\n        (run_dir / \"visuals\" / bucket).mkdir(parents=True, exist_ok=True)\n    (run_dir / \"seam_profile_plots\").mkdir(parents=True, exist_ok=True)\n    if len(dataset) > 0:\n        sample = dataset[0]\n        visuals = run_dir / \"visuals\" / \"median_10\"\n        for name in (\"input_rgb\", \"target\"):\n            Image.fromarray((sample[name].permute(1, 2, 0).numpy() * 255).astype(\"uint8\")).save(visuals / f\"{name}.png\")\n        Image.fromarray((((sample[\"target\"] - sample[\"input_rgb\"]).abs().mean(dim=0).numpy()) * 255).astype(\"uint8\")).save(visuals / \"error.png\")\n        Image.fromarray((torch.zeros_like(sample[\"target\"]).permute(1, 2, 0).numpy() * 255).astype(\"uint8\")).save(visuals / \"residual.png\")\n\n\nif __name__ == \"__main__\":\n    main()\n",
  "scripts/export_safetensors.py": "from __future__ import annotations\n\nimport argparse\nimport json\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nimport yaml\nfrom safetensors.torch import save_file\n\nfrom src.train.checkpoint import load_checkpoint\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\"--config\", default=\"configs/export_v1.yaml\")\n    args = parser.parse_args()\n    cfg = yaml.safe_load(Path(args.config).read_text(encoding=\"utf-8\"))\n    ckpt = load_checkpoint(Path(cfg[\"checkpoint\"]), map_location=\"cpu\")\n    export_root = Path(cfg[\"export_root\"])\n    export_root.mkdir(parents=True, exist_ok=True)\n    model_path = export_root / f\"{cfg['model_name']}.safetensors\"\n    save_file(ckpt[\"ema\"], str(model_path))\n    sidecar = {\n        \"model_name\": cfg[\"model_name\"],\n        \"schema_version\": 1,\n        \"architecture\": {\n            \"in_channels\": 5,\n            \"out_channels\": 3,\n            \"base_channels\": 32,\n            \"depth\": 4,\n            \"bottleneck\": {\"gap_film\": True, \"lf_branch\": True},\n            \"residual_cap_tanh_scale\": 0.3,\n            \"decay_window\": \"cosine\",\n        },\n        \"strip\": {\n            \"canonical_shape_chw\": [5, 1024, 256],\n            \"outer_width\": 128,\n            \"inner_width_default\": 128,\n            \"supported_inner_widths\": [96, 128, 160, 192],\n            \"seam_jitter_train_px\": 16,\n        },\n        \"preprocess\": {\n            \"rgb_range\": [0.0, 1.0],\n            \"expects_srgb_gamma\": True,\n            \"channels_order\": [\"R\", \"G\", \"B\", \"inner_mask\", \"distance_to_seam\"],\n        },\n        \"orientation\": {\"canonical\": \"vertical_outer_left\", \"train_rotation_aug\": True},\n        \"inference\": {\n            \"strength_range\": [0.0, 1.0],\n            \"strength_default\": cfg[\"strength_default\"],\n            \"hard_copy_outer\": True,\n            \"clamp_output\": [0.0, 1.0],\n        },\n        \"training\": {\n            \"dataset\": \"synthetic_only_v1\",\n            \"epochs\": 20,\n            \"ema_decay\": 0.999,\n            \"git_hash\": ckpt.get(\"git_hash\", \"nogit\"),\n            \"commit_date\": datetime.now(timezone.utc).isoformat(),\n        },\n    }\n    model_path.with_suffix(\".json\").write_text(json.dumps(sidecar, indent=2), encoding=\"utf-8\")\n\n\nif __name__ == \"__main__\":\n    main()\n",
  "scripts/verify_export.py": "from __future__ import annotations\n\nimport argparse\nimport json\nfrom pathlib import Path\n\nimport torch\nimport yaml\nfrom safetensors.torch import load_file\n\nfrom src.models.resunet import SeamResUNet\nfrom src.train.checkpoint import load_checkpoint\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\"--config\", default=\"configs/export_v1.yaml\")\n    args = parser.parse_args()\n    cfg = yaml.safe_load(Path(args.config).read_text(encoding=\"utf-8\"))\n    model_path = Path(cfg[\"export_root\"]) / f\"{cfg['model_name']}.safetensors\"\n    sidecar = json.loads(model_path.with_suffix(\".json\").read_text(encoding=\"utf-8\"))\n    a = sidecar[\"architecture\"]\n    state = load_file(str(model_path), device=\"cpu\")\n    model = SeamResUNet(\n        in_channels=a[\"in_channels\"],\n        out_channels=a[\"out_channels\"],\n        base_channels=a[\"base_channels\"],\n        residual_cap=a[\"residual_cap_tanh_scale\"],\n    )\n    model.load_state_dict(state)\n    model.eval()\n    ckpt = load_checkpoint(Path(cfg[\"checkpoint\"]), map_location=\"cpu\")\n    raw_model = SeamResUNet()\n    raw_model.load_state_dict(ckpt[\"ema\"])\n    x = torch.rand(1, 5, 1024, 256)\n    diff = (model(x) - raw_model(x)).abs().max().item()\n    if diff >= 1e-5:\n        raise RuntimeError(f\"verify_export failed: max diff={diff}\")\n    print({\"max_diff\": diff})\n\n\nif __name__ == \"__main__\":\n    main()\n"
}
for rel, text in PROJECT_FILES.items():
    path = PROJECT_ROOT / rel
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding='utf-8')
sys.path.insert(0, str(PROJECT_ROOT))
print('project files written:', len(PROJECT_FILES))


In [ ]:
# 6. VALIDATE BUNDLE LAYOUT
import json
required = [
    DATA_ROOT / 'manifests/strip_train_cache.jsonl',
    DATA_ROOT / 'manifests/strip_val_cache.jsonl',
    DATA_ROOT / 'outputs/strip_cache/train',
    DATA_ROOT / 'outputs/strip_cache/val',
]
for p in required:
    if not p.exists():
        raise FileNotFoundError(p)
print(json.dumps({
    'train_cache_dirs': len([p for p in (DATA_ROOT / 'outputs/strip_cache/train').iterdir() if p.is_dir()]),
    'val_cache_dirs': len([p for p in (DATA_ROOT / 'outputs/strip_cache/val').iterdir() if p.is_dir()]),
}, ensure_ascii=False))


In [ ]:
# 7. BUILD RUNTIME CONFIGS (обязателен перед 10, либо 10 сам вызовет тот же скрипт, если yml нет)
import subprocess, sys
subprocess.check_call(
    [sys.executable, str(PROJECT_ROOT / 'scripts' / 'write_colab_runtime_yamls.py'),
     '--project-root', str(PROJECT_ROOT), '--data-root', str(DATA_ROOT),
     '--train-batch-size', str(TRAIN_BATCH_SIZE), '--train-epochs', str(TRAIN_EPOCHS),
     '--train-num-workers', str(TRAIN_NUM_WORKERS), '--primary-checkpoint', PRIMARY_CHECKPOINT],
    cwd=str(PROJECT_ROOT),
)
print('runtime configs ->', PROJECT_ROOT / 'runtime_configs')


In [ ]:
# 8. OPTIONAL RESUME FROM DRIVE LAST CHECKPOINT
import shutil
resume_path = None
DRIVE_CKPT_DIR.mkdir(parents=True, exist_ok=True)
drive_last = DRIVE_CKPT_DIR / 'last.pt'
if drive_last.exists():
    LOCAL_CHECKPOINTS.mkdir(parents=True, exist_ok=True)
    local_resume = LOCAL_CHECKPOINTS / 'resume_last.pt'
    shutil.copy2(drive_last, local_resume)
    resume_path = local_resume
print('resume_path =', resume_path)


In [ ]:
# 9. BACKGROUND SYNC TO DRIVE (eval/exports on Drive only after you run cells 11–12; checkpoints sync during train if this cell ran before 10)
import threading, time, shutil
SYNC_STOP = threading.Event()
for p in [DRIVE_RUN_DIR, DRIVE_CKPT_DIR, DRIVE_EVAL_DIR, DRIVE_EXPORT_DIR, DRIVE_LOG_DIR]:
    p.mkdir(parents=True, exist_ok=True)
def sync_tree(src: Path, dst: Path) -> int:
    n = 0
    if not src.exists():
        return 0
    for path in src.rglob('*'):
        if not path.is_file():
            continue
        rel = path.relative_to(src)
        out = dst / rel
        out.parent.mkdir(parents=True, exist_ok=True)
        if not out.exists() or out.stat().st_mtime < path.stat().st_mtime or out.stat().st_size != path.stat().st_size:
            try:
                shutil.copy2(path, out)
                n += 1
            except OSError as e:
                print('sync error', path, '->', out, e, flush=True)
    return n
def sync_all_to_drive() -> None:
    a = sync_tree(LOCAL_CHECKPOINTS, DRIVE_CKPT_DIR)
    b = sync_tree(LOCAL_EVAL, DRIVE_EVAL_DIR)
    c = sync_tree(LOCAL_EXPORTS, DRIVE_EXPORT_DIR)
    d = sync_tree(LOCAL_LOGS, DRIVE_LOG_DIR)
    print('sync tick: ckpt', a, 'eval', b, 'export', c, 'logs', d, '->', DRIVE_RUN_DIR, flush=True)
def sync_loop():
    while True:
        sync_all_to_drive()
        if SYNC_STOP.wait(SYNC_INTERVAL_SEC):
            break
sync_all_to_drive()
sync_thread = threading.Thread(target=sync_loop, daemon=True)
sync_thread.start()
print('background sync started (first tick already ran)')


In [ ]:
# 10. TRAIN
import os, sys, subprocess, threading
if not (PROJECT_ROOT / 'runtime_configs' / 'train.yaml').is_file():
    print('Нет runtime_configs — запускаю scripts/write_colab_runtime_yamls.py (как в ячейке 7)…')
    subprocess.check_call(
        [sys.executable, str(PROJECT_ROOT / 'scripts' / 'write_colab_runtime_yamls.py'),
         '--project-root', str(PROJECT_ROOT), '--data-root', str(DATA_ROOT),
         '--train-batch-size', str(TRAIN_BATCH_SIZE), '--train-epochs', str(TRAIN_EPOCHS),
         '--train-num-workers', str(TRAIN_NUM_WORKERS), '--primary-checkpoint', PRIMARY_CHECKPOINT],
        cwd=str(PROJECT_ROOT),
    )
env = os.environ.copy()
env['PYTHONPATH'] = str(PROJECT_ROOT)
env['PYTHONUNBUFFERED'] = '1'
cmd = [sys.executable, '-u', '-m', 'scripts.train_resunet', '--config', str(PROJECT_ROOT / 'runtime_configs/train.yaml')]
if resume_path is not None:
    cmd += ['--resume', str(resume_path)]
print('TRAIN CMD:', ' '.join(map(str, cmd)))
print('Дальше: loading, train_start, epoch_begin, train_iter_begin (затем пауза до первого train_step — нормально: 1-й батч + воркеры + GPU).')
def _stream_cmd(cmd, cwd, env):
    # Colab: inherited stdio often shows nothing; PIPE + a thread that prints to the cell is reliable. Main thread only p.wait() (not read()).
    p = subprocess.Popen(
        cmd, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    def _pump():
        if p.stdout is not None:
            for line in p.stdout:
                print(line, end='', flush=True)
    t = threading.Thread(target=_pump, daemon=True)
    t.start()
    rc = p.wait()
    t.join(timeout=30)
    if rc != 0:
        print(f'\n[exit {rc}] Причина — в Traceback/ошибке ВЫШЕ. CalledProcessError — только итог.', flush=True)
        raise subprocess.CalledProcessError(rc, cmd)
_stream_cmd(cmd, str(PROJECT_ROOT), env)


In [ ]:
# 10b. TensorBoard (после старта train в 10; можно перезапускать)
import subprocess, sys, time, shutil
from google.colab import output

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "protobuf<5", "tensorboard"], check=True)

LOGDIR = PROJECT_ROOT / "outputs" / "logs" / "tensorboard"
LOGDIR.mkdir(parents=True, exist_ok=True)
subprocess.Popen(
    [sys.executable, "-m", "tensorboard.main", "--logdir", str(LOGDIR), "--port", "6006", "--bind_all"],
    start_new_session=True,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT,
)
time.sleep(5)
DRIVE_TB = DRIVE_RUN_DIR / "tensorboard"
DRIVE_TB.mkdir(parents=True, exist_ok=True)
for path in LOGDIR.rglob("*"):
    if path.is_file():
        rel = path.relative_to(LOGDIR)
        out = DRIVE_TB / rel
        out.parent.mkdir(parents=True, exist_ok=True)
        try:
            shutil.copy2(path, out)
        except OSError:
            pass
print("LOGDIR =", LOGDIR, "| копия на Drive:", DRIVE_TB)
output.serve_kernel_port_as_window(6006, path="/")
print("Если встроенный просмотр серый — открой ссылку в новой вкладке.")

In [ ]:
# 11. EVAL
import os, sys, subprocess
if '_stream_cmd' not in globals():
    import threading
    def _stream_cmd(cmd, cwd, env):
        p = subprocess.Popen(
            cmd, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
        )
        def _pump():
            if p.stdout is not None:
                for line in p.stdout:
                    print(line, end='', flush=True)
        t = threading.Thread(target=_pump, daemon=True)
        t.start()
        rc = p.wait()
        t.join(timeout=30)
        if rc != 0:
            print(f'\n[exit {rc}] See Traceback above.', flush=True)
            raise subprocess.CalledProcessError(rc, cmd)
env = os.environ.copy()
env['PYTHONPATH'] = str(PROJECT_ROOT)
env['PYTHONUNBUFFERED'] = '1'
cmd = [sys.executable, '-u', '-m', 'scripts.run_eval', '--config', str(PROJECT_ROOT / 'runtime_configs/eval.yaml')]
print('EVAL CMD:', ' '.join(map(str, cmd)))
_stream_cmd(cmd, str(PROJECT_ROOT), env)


In [ ]:
# 12. EXPORT + VERIFY
import os, sys, subprocess
if '_stream_cmd' not in globals():
    import threading
    def _stream_cmd(cmd, cwd, env):
        p = subprocess.Popen(
            cmd, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
        )
        def _pump():
            if p.stdout is not None:
                for line in p.stdout:
                    print(line, end='', flush=True)
        t = threading.Thread(target=_pump, daemon=True)
        t.start()
        rc = p.wait()
        t.join(timeout=30)
        if rc != 0:
            print(f'\n[exit {rc}] See Traceback above.', flush=True)
            raise subprocess.CalledProcessError(rc, cmd)
env = os.environ.copy()
env['PYTHONPATH'] = str(PROJECT_ROOT)
env['PYTHONUNBUFFERED'] = '1'
export_cmd = [sys.executable, '-u', '-m', 'scripts.export_safetensors', '--config', str(PROJECT_ROOT / 'runtime_configs/export.yaml')]
verify_cmd = [sys.executable, '-u', '-m', 'scripts.verify_export', '--config', str(PROJECT_ROOT / 'runtime_configs/export.yaml')]
print('EXPORT CMD:', ' '.join(map(str, export_cmd)))
_stream_cmd(export_cmd, str(PROJECT_ROOT), env)
print('VERIFY CMD:', ' '.join(map(str, verify_cmd)))
_stream_cmd(verify_cmd, str(PROJECT_ROOT), env)


In [ ]:
# 13. FINAL SYNC + SUMMARY
SYNC_STOP.set()
sync_all_to_drive()
print('Drive checkpoint files:', sorted(p.name for p in DRIVE_CKPT_DIR.glob('*')))
print('Drive export files:', sorted(p.name for p in DRIVE_EXPORT_DIR.glob('*')))
print('Drive eval runs:', sorted(p.name for p in DRIVE_EVAL_DIR.glob('*')))
